<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.es/cap09/cap09.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Práctica con Ejercicios de Programación**

La presente lista de **Ejercicios de Programación (EP)** consolida las formulaciones teóricas presentadas a lo largo del Capítulo 9 — Aprendizaje Profundo para Visión por Computador — mediante una ruta práctica aplicada. A diferencia del entrenamiento de redes neuronales completas con PyTorch, que requiere tiempo de ejecución y, en ocasiones, GPU, los EP de este capítulo aislan las **magnitudes intermedias** de un *pipeline* real de aprendizaje profundo —la salida de una única capa convolucional, el resultado de una operación de *pooling*, el conteo de parámetros entrenables de una arquitectura, la superposición entre cajas delimitadoras candidatas, la calidad de una máscara de segmentación y el filtro de supresión de no máximos—, permitiendo validar manualmente cada etapa del razonamiento sin depender de bibliotecas de aprendizaje automático ni de entrenamiento real.

El encadenamiento de los ejercicios reproduce el flujo conceptual del capítulo y crece en dificultad a cada paso: se comienza con el cálculo manual de la salida de una **capa convolucional aprendida** (🟢), a partir de un *kernel* y un sesgo ya entrenados; se avanza hacia la operación de ***pooling*** (🟢, máximo y promedio), que reduce la resolución espacial entre bloques convolucionales; se continúa con el **conteo de parámetros entrenables** (🟡) de una arquitectura CNN completa, evidenciando por qué el compartimiento de pesos vuelve a estas redes mucho más económicas que una capa totalmente conectada equivalente; se profundiza en el cálculo de **Intersección sobre Unión (IoU)** y en la **Supresión de No Máximos (NMS)** (🟡), etapa de posprocesamiento común a detectores como Faster R-CNN y YOLO; se sigue con la **evaluación de máscaras de segmentación** (🟠) con las mismas métricas de IoU y Dice usadas para comparar la U-Net con la línea base morfológica clásica; y se concluye con un ***pipeline* integrado** (🔴), que une la salida de un detector de objetos (después de NMS) con una medición del mundo real por referencia de escala —el mismo principio de la fotogrametría estudiada en la integración final del capítulo.

Siempre que tenga sentido, cada ejercicio señala métodos de la biblioteca didáctica `morph.py` (la misma usada a lo largo del capítulo, importada como `mm`) que resuelven una etapa del problema o que sirven de referencia para verificar tus propios cálculos —sin, no obstante, sustituir el razonamiento que debes implementar.

> ### ❗ Directrices para la Resolución de los Ejercicios de Programación
>
> En todos los ejercicios de este capítulo, las etapas de discretización o redondeo numérico deben emplear el redondeo estándar al entero más cercano (*round half away from zero*), mitigando ambigüedades en valores con fracción exactamente igual a $0{,}5$. Salvo indicación explícita en contrario: (i) la operación de "convolución" sigue la convención adoptada por los *frameworks* de aprendizaje profundo —**correlación cruzada**, sin inversión espacial del *kernel*, exactamente como se presentó en la Sección "Capa Convolucional"; (ii) el relleno (*padding*) se realiza con ceros; (iii) las cajas delimitadoras se especifican en el formato esquina-a-esquina $(x_1, y_1, x_2, y_2)$, con $x_1 < x_2$ e $y_1 < y_2$; y (iv) los vectores/matrices siguen indexación desde $0$, con la convención `[fila][columna]` para estructuras bidimensionales.

### 🎯 Objetivo de este Cuaderno

El cuaderno permite desarrollar, validar, organizar y probar soluciones de **Ejercicios de Programación (EPs)** en entornos interactivos, como Colab, con los mismos casos de prueba de Moodle, copiándolos allí solo al momento de registrar la nota oficial.

#### *Descargar*

Descargue `morph.py` y `testsuite.py` ejecutando la celda a continuación:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Ejecutando los Tests
Para evaluar los tests, ejecute `TestSuite("EP09_01.extensión").run()` en una nueva celda, reemplazando la extensión por la del lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema descarga los casos de prueba desde GitHub, ejecuta el programa y calcula la nota automáticamente.

Para probar código Python directamente, sin guardar archivo, use `run_code(codigo)` pasando el código como *string* en una variable `codigo`:

```python
codigo = """
# ... su código aquí ...
"""
TestSuite("EP09_01").run_code(codigo)
```

### EP09_01 🟢 Convolución 2D Manual (*Forward* de una Capa Aprendida)

PyTorch, presentado en este capítulo, ejecuta `nn.Conv2d(x)` en una única llamada — pero detrás de ella solo está la correlación cruzada entre un *kernel* (ya entrenado) y una vecindad de la entrada, seguida de la suma de un sesgo y de una activación, exactamente como se formalizó en la Sección "Capa Convolucional". La diferencia esencial respecto a la convolución de *kernels* fijos del Capítulo 3 es que, aquí, los valores del *kernel* y del sesgo **ya vienen listos** (como si hubieran sido aprendidos por gradiente), y te corresponde a ti reproducir manualmente la pasada directa (*forward pass*) que el *framework* ejecuta internamente.

Antes de entrenar una CNN real, se te ha encargado implementar esta pasada directa desde cero, para una única capa convolucional con un único canal de entrada y un único filtro de salida, incluyendo soporte para *padding* y *stride* arbitrarios.

#### 📋 Directrices de Implementación

1. **Entrada:** Leer las dimensiones $H \times W$ del mapa de características de entrada y, a continuación, sus $H \times W$ valores reales.
   
2. ***Kernel* y sesgo:** Leer las dimensiones $k_h \times k_w$ del *kernel* (ya entrenado), sus valores reales, y el sesgo $b$ (real, escalar).
   
3. **Hiperparámetros:** Leer el *padding* $p$ (entero, número de ceros añadidos en cada borde) y el *stride* $s$ (entero, paso del deslizamiento).
   
4. **Relleno:** Añadir $p$ ceros en cada uno de los cuatro bordes del mapa de entrada antes de la correlación.
   
5. **Correlación cruzada:** Para cada posición de salida $(i, j)$, calcular
   $$
   z(i,j) = b + \sum_{u=0}^{k_h-1} \sum_{v=0}^{k_w-1} K(u,v) \cdot X_{pad}(i \cdot s + u,\; j \cdot s + v),
   $$
   recorriendo la entrada **sin** invertir el *kernel* (convención de los *frameworks* de aprendizaje profundo, diferente de la convolución matemática clásica).

6. **Activación:** Aplicar ReLU a cada valor: $a(i,j) = \max(0, z(i,j))$.

7. **Dimensiones de salida:** $O_h = \lfloor (H + 2p - k_h)/s \rfloor + 1$ y $O_w = \lfloor (W + 2p - k_w)/s \rfloor + 1$.

8. **Salida:** Imprimir $O_h$ y $O_w$ en la primera línea, seguidos de $O_h$ líneas con $O_w$ valores reales cada una (el mapa de características de salida, ya con ReLU aplicada), formateados con 4 decimales.

#### 📌 Restricciones Computacionales

* **Un canal de entrada, un filtro de salida:** no es necesario manejar múltiples canales ni múltiples filtros en esta versión simplificada.
* **Sin inversión del *kernel*:** implementa correlación cruzada, no la convolución matemática clásica con *kernel* invertido — es esa la operación que PyTorch (y la mayoría de los *frameworks*) llama "convolución".
* **Relleno con ceros:** los $p$ píxeles añadidos en cada borde valen siempre $0$.
* **Formato:** todos los valores de salida deben tener exactamente 4 decimales, incluso cuando el valor es entero (ej.: `2.0000`).

#### 🧠 Fundamentación Teórica

| Elemento | Papel en la capa convolucional |
|---|---|
| *Kernel* $K$ | Parámetros aprendidos por gradiente, análogos a los coeficientes de un filtro fijo del Capítulo 3, pero ajustados por retropropagación |
| Sesgo $b$ | Desplazamiento aprendido, sumado tras la correlación — permite que la neurona "se active" incluso con entrada nula |
| *Padding* | Controla la dimensión espacial de salida y evita la pérdida de información en los bordes en cada capa |
| *Stride* | Controla el paso del desplazamiento; valores $> 1$ reducen la resolución espacial, como una forma de submuestreo integrado en la propia convolución |
| ReLU | Introduce no linealidad tras la combinación lineal, exactamente como en la Sección "Función de Activación" |

#### 🧩 Métodos del `morph.py` que pueden ayudar

* `mm.readImg(h, w, dtype='float')` — lee directamente una matriz $h \times w$ de valores reales de la entrada estándar, ahorrando el *parseo* manual del mapa de características y del *kernel*.
* `mm.correlacao0(f, kernel, bias)` — implementa la misma suma de correlación cruzada + sesgo que vas a calcular a mano, pero **sin** soporte para *padding* o *stride*, y convierte el resultado a `uint8` (trunca valores negativos y decimales). Puede servir como referencia conceptual o para comprobar el caso más simple ($p=0$, $s=1$), pero no sustituye tu implementación completa — que debe preservar signo, decimales, *padding*, *stride* y ReLU.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $H$ y $W$.
* Siguientes $H$ líneas: $W$ valores reales cada una (mapa de entrada).
* Siguiente línea: Enteros $k_h$ y $k_w$.
* Siguientes $k_h$ líneas: $k_w$ valores reales cada una (*kernel*).
* Siguiente línea: Real $b$ (sesgo).
* Siguiente línea: Enteros $p$ y $s$.

**Salida:**

* Línea 1: Enteros $O_h$ y $O_w$.
* Siguientes $O_h$ líneas: $O_w$ valores reales cada una, con 4 decimales.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 1<br>1 1<br>-2<br>0 1 | 2 2<br>2.0000 3.0000<br>0.0000 2.0000 | *Padding* 0, *stride* 1: salida $2\times2$ sin relleno. |
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 0<br>0 1<br>0<br>1 2 | 2 2<br>1.0000 0.0000<br>1.0000 2.0000 | *Padding* 1, *stride* 2: entrada rellenada con ceros antes de la correlación. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0901" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Convolución 2D Manual</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 correlación cruzada + sesgo + ReLU</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Entrada 4×4 fija, kernel 2×2 fijo (resaltado en azul) &mdash; ajuste <em>padding</em> (p), <em>stride</em> (s) y sesgo (b), exactamente los parámetros que EP09_01 pide en la entrada, y observa cómo cambian el tamaño y los valores de la salida.</p>

    <div style="display:flex;gap:24px;justify-content:center;flex-wrap:wrap;margin-bottom:6px;">
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Padding (p)</div>
        <div id="ep0901_pad_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Stride (s)</div>
        <div id="ep0901_stride_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Sesgo (b)</div>
        <input id="ep0901_bias" type="number" step="0.5" value="0.5" style="width:70px;font-family:monospace;text-align:center;border:1px solid #ccc;border-radius:6px;padding:3px;">
      </div>
    </div>

    <div id="ep0901_formula" style="text-align:center;font-family:monospace;font-size:11px;color:#8a6d3b;background:#fcf5e6;border:1px solid #f0e2bf;border-radius:8px;padding:6px 10px;margin:10px auto 16px;max-width:520px;"></div>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Posición de salida (i,j)</label>
        <span id="ep0901_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0901_sl" style="width:100%;accent-color:#2980b9;" max="8" min="0" step="1" type="range" value="0">
    </div>

    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;align-items:flex-start;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Entrada X rellenada (con padding)</div>
        <div id="ep0901_grid" style="display:grid;gap:3px;"></div>
        <div style="display:flex;gap:10px;justify-content:center;margin-top:8px;font-size:9px;color:#888;">
          <span><span style="display:inline-block;width:9px;height:9px;background:#f3f4f6;border:1px solid #e5e7eb;border-radius:2px;vertical-align:middle;"></span> original</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#f5f5f5;border:1px dashed #ccc;border-radius:2px;vertical-align:middle;"></span> padding (0)</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#fff3cd;border:2px solid #f0ad4e;border-radius:2px;vertical-align:middle;"></span> ventana actual</span>
        </div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Kernel K (2×2)</div>
        <div id="ep0901_kernel" style="display:grid;grid-template-columns:repeat(2,44px);gap:3px;"></div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Salida Y = ReLU(X⊛K + b)</div>
        <div id="ep0901_out" style="display:grid;gap:3px;"></div>
      </div>
    </div>

    <div style="margin-top:14px;text-align:center;">
      <button id="ep0901_reset" style="font-family:sans-serif;font-size:11px;color:#5e5a4a;background:#f3efe6;border:1px solid #e0d7c4;border-radius:20px;padding:4px 14px;cursor:pointer;">↺ reiniciar exploración</button>
    </div>
    <div id="ep0901_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
    <p style="font-size:10px;color:#999;margin-top:10px;text-align:center;">💡 Cada posición del control deslizante revela una celda de la matriz de salida. Recorre todas las posiciones para completar el mapa de salida. Cambiar p, s o b reinicia la exploración, porque el mapa de salida cambia de tamaño y/o de valores.</p>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var H = 4, W = 4, kh = 2, kw = 2;
    var X = [[1,3,2,0],[0,1,4,1],[2,0,1,3],[1,2,0,1]];
    var K = [[1,0],[0,-1]];

    var state = { p: 0, s: 1, bias: 0.5 };
    var visited = {};

    var slEl = root.querySelector('#ep0901_sl');
    var vlEl = root.querySelector('#ep0901_vl');
    var gridEl = root.querySelector('#ep0901_grid');
    var kernelEl = root.querySelector('#ep0901_kernel');
    var outEl = root.querySelector('#ep0901_out');
    var dbg = root.querySelector('#ep0901_debug');
    var formulaEl = root.querySelector('#ep0901_formula');
    var resetBtn = root.querySelector('#ep0901_reset');
    var padBtnsEl = root.querySelector('#ep0901_pad_btns');
    var strideBtnsEl = root.querySelector('#ep0901_stride_btns');
    var biasInput = root.querySelector('#ep0901_bias');

    kernelEl.innerHTML = '';
    for(var u=0; u<kh; u++) for(var v=0; v<kw; v++){
      var kd = document.createElement('div');
      kd.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;background:#bbdefb;border:1px solid #64b5f6;border-radius:6px;font-family:monospace;font-weight:bold;color:#0d47a1;';
      kd.textContent = K[u][v];
      kernelEl.appendChild(kd);
    }

    function btnStyle(active){
      return 'font-family:monospace;font-size:12px;font-weight:bold;width:34px;height:30px;border-radius:8px;cursor:pointer;' +
        (active ? 'background:#2980b9;color:white;border:1px solid #2471a3;' : 'background:#f3f4f6;color:#555;border:1px solid #ddd;');
    }

    function buildButtons(container, values, current, onPick){
      container.innerHTML = '';
      values.forEach(function(val){
        var b = document.createElement('button');
        b.textContent = val;
        b.style.cssText = btnStyle(val === current);
        b.addEventListener('click', function(){ onPick(val); });
        container.appendChild(b);
      });
    }

    function buildPadded(p){
      var size = H + 2*p;
      var Xp = [];
      for(var r=0; r<size; r++){
        var row = [];
        for(var c=0; c<size; c++){
          var origR = r-p, origC = c-p;
          var isPad = !(origR>=0 && origR<H && origC>=0 && origC<W);
          row.push({ val: isPad ? 0 : X[origR][origC], pad: isPad });
        }
        Xp.push(row);
      }
      return Xp;
    }

    function computeAll(p, s, bias){
      var Xp = buildPadded(p);
      var size = H + 2*p;
      var Oh = Math.floor((size - kh)/s) + 1;
      var Ow = Math.floor((size - kw)/s) + 1;
      var vals = [];
      for(var i=0; i<Oh; i++){
        vals.push([]);
        for(var j=0; j<Ow; j++){
          var soma = 0;
          for(var u=0; u<kh; u++) for(var v=0; v<kw; v++) soma += K[u][v]*Xp[i*s+u][j*s+v].val;
          var z = soma + bias;
          var a = Math.max(0, z);
          vals[i].push({ soma: soma, z: z, a: a });
        }
      }
      return { Xp: Xp, size: size, Oh: Oh, Ow: Ow, vals: vals };
    }

    var model = null;

    function rebuildModel(resetPosition){
      model = computeAll(state.p, state.s, state.bias);
      gridEl.style.gridTemplateColumns = 'repeat(' + model.size + ', ' + Math.min(44, Math.floor(360/model.size)) + 'px)';
      outEl.style.gridTemplateColumns = 'repeat(' + model.Ow + ', 44px)';
      var maxPos = model.Oh*model.Ow - 1;
      slEl.max = maxPos;
      if(resetPosition){ slEl.value = 0; visited = {}; }
      else if(parseInt(slEl.value) > maxPos){ slEl.value = maxPos; }
      formulaEl.textContent = 'O_h = ⌊(' + H + ' + 2·' + state.p + ' − ' + kh + ')/' + state.s + '⌋ + 1 = ' + model.Oh +
        '    O_w = ⌊(' + W + ' + 2·' + state.p + ' − ' + kw + ')/' + state.s + '⌋ + 1 = ' + model.Ow;
      render();
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/model.Ow), j = pos % model.Ow;
      var key = i+','+j;
      visited[key] = true;
      vlEl.textContent = '('+i+','+j+')';

      var winRowStart = i*state.s, winColStart = j*state.s;

      gridEl.innerHTML = '';
      var cellPx = Math.min(44, Math.floor(360/model.size));
      for(var r=0; r<model.size; r++){
        for(var c=0; c<model.size; c++){
          var cell = model.Xp[r][c];
          var dentroJanela = (r>=winRowStart && r<winRowStart+kh && c>=winColStart && c<winColStart+kw);
          var d = document.createElement('div');
          var base = 'width:'+cellPx+'px;height:'+cellPx+'px;display:flex;align-items:center;justify-content:center;border-radius:5px;font-family:monospace;font-size:11px;';
          if(dentroJanela){
            base += 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;';
          } else if(cell.pad){
            base += 'background:#f5f5f5;border:1px dashed #ccc;color:#bbb;';
          } else {
            base += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;';
          }
          d.style.cssText = base;
          d.textContent = cell.val;
          gridEl.appendChild(d);
        }
      }

      var cur = model.vals[i][j];

      outEl.innerHTML = '';
      for(var oi=0; oi<model.Oh; oi++){
        for(var oj=0; oj<model.Ow; oj++){
          var isCurrent = (oi===i && oj===j);
          var wasVisited = !!visited[oi+','+oj];
          var od = document.createElement('div');
          var style = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:11px;font-weight:bold;';
          if(isCurrent){
            style += 'background:#fff3cd;border:2px solid #f0ad4e;color:#7a5c00;';
          } else if(wasVisited){
            style += 'background:#e8f5e9;border:1px solid #a5d6a7;color:#2e7d32;';
          } else {
            style += 'background:#f3f4f6;border:1px dashed #ccc;color:#bbb;';
          }
          od.style.cssText = style;
          od.textContent = wasVisited ? model.vals[oi][oj].a.toFixed(2) : '?';
          outEl.appendChild(od);
        }
      }

      var totalVisitadas = Object.keys(visited).length;
      var totalPos = model.Oh*model.Ow;
      dbg.textContent = 'janela em ('+i+','+j+'), topo-esquerda em X_pad('+winRowStart+','+winColStart+')  |  soma(X⊙K)='+cur.soma.toFixed(2)+'  +  viés='+state.bias.toFixed(2)+'  =  z='+cur.z.toFixed(2)+'  →  ReLU(z)='+cur.a.toFixed(4)+'   ['+totalVisitadas+'/'+totalPos+' posições exploradas]';
    }

    function setPadding(val){
      state.p = val;
      buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
      buildButtons(strideBtnsEl, [1,2], state.s, setStride);
      rebuildModel(true);
    }
    function setStride(val){
      state.s = val;
      buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
      buildButtons(strideBtnsEl, [1,2], state.s, setStride);
      rebuildModel(true);
    }
    buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
    buildButtons(strideBtnsEl, [1,2], state.s, setStride);

    biasInput.addEventListener('change', function(){
      var v = parseFloat(biasInput.value);
      state.bias = isNaN(v) ? 0 : v;
      rebuildModel(true);
    });

    resetBtn.addEventListener('click', function(){
      visited = {};
      slEl.value = 0;
      render();
    });
    slEl.addEventListener('input', render);

    rebuildModel(true);
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0901');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.1:** Simulador EP09_01: Convolución 2D Manual (correlación cruzada + sesgo + ReLU, con *padding* y *stride* ajustables)


<figure id="fig-09-sim-ep0901">
  <img src="imagens/fig-09-sim-ep0901.png" alt=" Simulador EP09_01: Convolución 2D Manual (correlación cruzada + sesgo + ReLU, con *padding* y *stride* ajustables) " style="max-width:80%" />
  <figcaption><strong>Figura 9.1:</strong>  Simulador EP09_01: Convolución 2D Manual (correlación cruzada + sesgo + ReLU, con *padding* y *stride* ajustables) </figcaption>
</figure>

In [ ]:
%%writefile EP09_01.py
# Código Python

In [ ]:
TestSuite("EP09_01.py").run()

### EP09_02 🟢 *Pooling* Manual (Máximo y Media)

Entre bloques convolucionales, la arquitectura típica de una CNN intercala capas de ***pooling***, que reducen la resolución espacial del mapa de características sin introducir nuevos parámetros entrenables — a diferencia de la convolución, el *pooling* no tiene pesos: solo resume cada ventana de la entrada a un único valor, mediante un máximo o una media, exactamente como se formalizó en la Sección "*Pooling*".

Se le ha encargado implementar esta operación a partir de una ventana deslizante cuadrada, sin superposición parcial en los bordes (solo ventanas completas), soportando los dos tipos más comunes: `max` (preserva el valor más sobresaliente, típicamente usado para retener bordes y texturas fuertes) y `avg` (suaviza la región, preservando información de intensidad media).

#### 📋 Directrices de Implementación

1. **Entrada:** Leer las dimensiones $H \times W$ del mapa de características de entrada y sus $H \times W$ valores reales.
2. **Ventana:** Leer los enteros $k$ (tamaño de la ventana cuadrada $k \times k$) y $s$ (*stride*).
3. **Tipo:** Leer una *string*, `max` o `avg`, indicando el tipo de *pooling*.
4. **Sin relleno:** Esta operación **no** utiliza *padding*; las ventanas que sobrepasarían el borde de la entrada se descartan.
5. **Cálculo:** Para cada posición de salida $(i,j)$, calcular el máximo o la media de los $k \times k$ valores de la ventana correspondiente, comenzando en $(i \cdot s,\, j \cdot s)$.
6. **Dimensiones de salida:** $O_h = \lfloor (H - k)/s \rfloor + 1$ y $O_w = \lfloor (W - k)/s \rfloor + 1$.
7. **Salida:** Imprimir $O_h$ y $O_w$ en la primera línea, seguidos de $O_h$ líneas con $O_w$ valores reales cada una, formateados con 4 decimales.

#### 📌 Restricciones Computacionales

* **Ventana cuadrada:** $k \times k$, sin soporte para ventanas rectangulares en esta versión.
* **Sin *padding*:** solo se consideran ventanas completamente contenidas en la entrada — las dimensiones que "sobran" simplemente se descartan.
* **`avg` usa división real:** la media es siempre $\text{suma}/k^2$, incluso cuando el resultado tiene muchos decimales — redondee solo en el formato final, conforme a la directriz general del capítulo.
* **Formato:** todos los valores de salida con exactamente 4 decimales.

#### 🧠 Fundamentación Teórica

| Elemento | Rol en la arquitectura |
|---|---|
| *Pooling* máximo | Preserva la activación más fuerte de la ventana; común después de capas convolucionales para retener bordes y texturas sobresalientes |
| *Pooling* medio | Suaviza la región, preservando la intensidad media; común en capas finales (*global average pooling*) |
| Ausencia de parámetros | Diferencia el *pooling* de la convolución: reduce la resolución espacial sin costo adicional de entrenamiento |
| Reducción de resolución | Contribuye a la invariancia a pequeñas traslaciones y a la reducción del costo computacional de las capas siguientes |

#### 🧩 Métodos de `morph.py` que pueden ayudar

El `morph.py` no implementa *pooling* con submuestreo directamente, pero dos familias de operaciones muestran la misma idea desde otra óptica, útil para verificar su intuición:

* `mm.dil(f, Bc)` / `mm.dil0(f, B)` — dilatación morfológica: reemplaza cada píxel por el **máximo** de su vecindad definida por el elemento estructurante $B$ (ej.: `mm.sebox(n)` para una ventana $(2n+1)\times(2n+1)$). Es, conceptualmente, un "*max-pooling* sin submuestreo" (produce una imagen del mismo tamaño, en lugar de reducida).
* `mm.blur(f, N)` — suavizado por media en una ventana $N \times N$, análoga al *avg-pooling*, también sin reducción de resolución.
* `mm.readImg(h, w, dtype='float')` — útil para leer el mapa de entrada en punto flotante.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $H$ y $W$.
* Siguientes $H$ líneas: $W$ valores reales cada una.
* Siguiente línea: Enteros $k$ y $s$.
* Siguiente línea: `max` o `avg`.

**Salida:**

* Línea 1: Enteros $O_h$ y $O_w$.
* Siguientes $O_h$ líneas: $O_w$ valores reales cada una, con 4 decimales.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>max | 2 2<br>6.0000 4.0000<br>4.0000 5.0000 | *Pooling* máximo, ventana $2\times2$, *stride* 2. |
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>avg | 2 2<br>3.7500 2.2500<br>2.2500 2.2500 | *Pooling* medio sobre las mismas ventanas. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0902" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Pooling Manual</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 senza padding, finestre complete</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ingresso 4×4 fisso &mdash; regola la dimensione della finestra (k), lo stride (s) e il tipo, esattamente i parametri che EP09_02 legge in ingresso, e osserva come cambiano la dimensione e i valori dell'uscita.</p>

    <div style="display:flex;gap:24px;justify-content:center;flex-wrap:wrap;margin-bottom:6px;">
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Finestra (k)</div>
        <div id="ep0902_k_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Stride (s)</div>
        <div id="ep0902_s_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Tipo</div>
        <div style="display:flex;gap:8px;">
          <button id="ep0902_max" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #f0ad4e;background:#fff3cd;color:#7a5c00;font-weight:bold;font-size:11px;">max</button>
          <button id="ep0902_avg" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #ddd;background:#f3f4f6;color:#555;font-weight:bold;font-size:11px;">avg</button>
        </div>
      </div>
    </div>

    <div id="ep0902_formula" style="text-align:center;font-family:monospace;font-size:11px;color:#8a6d3b;background:#fcf5e6;border:1px solid #f0e2bf;border-radius:8px;padding:6px 10px;margin:10px auto 16px;max-width:520px;"></div>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Posizione di uscita (i,j)</label>
        <span id="ep0902_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0902_sl" style="width:100%;accent-color:#2980b9;" max="3" min="0" step="1" type="range" value="0">
    </div>

    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;align-items:flex-start;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Ingresso X (4×4)</div>
        <div id="ep0902_grid" style="display:grid;grid-template-columns:repeat(4,44px);gap:3px;"></div>
        <div style="display:flex;gap:10px;justify-content:center;margin-top:8px;font-size:9px;color:#888;">
          <span><span style="display:inline-block;width:9px;height:9px;background:#f3f4f6;border:1px solid #e5e7eb;border-radius:2px;vertical-align:middle;"></span> fuori dalla finestra</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#fff3cd;border:2px solid #f0ad4e;border-radius:2px;vertical-align:middle;"></span> finestra attuale</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#eee;border:1px dashed #bbb;border-radius:2px;vertical-align:middle;"></span> scartato (avanzo)</span>
        </div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Uscita Y (pooling)</div>
        <div id="ep0902_out" style="display:grid;gap:3px;"></div>
      </div>
    </div>

    <div style="margin-top:14px;text-align:center;">
      <button id="ep0902_reset" style="font-family:sans-serif;font-size:11px;color:#5e5a4a;background:#f3efe6;border:1px solid #e0d7c4;border-radius:20px;padding:4px 14px;cursor:pointer;">↺ riavvia esplorazione</button>
    </div>
    <div id="ep0902_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
    <p style="font-size:10px;color:#999;margin-top:10px;text-align:center;">💡 Ogni posizione dello slider rivela una cella della matrice di uscita. Le celle grigio-tratteggiate nell'ingresso sono "avanzi" che nessuna finestra raggiunge &mdash; nota come ciò accade quando (H&minus;k) non è multiplo di s. Cambiare k, s o il tipo riavvia l'esplorazione.</p>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var H = 4, W = 4;
    var X = [[1,3,2,4],[5,6,1,2],[2,1,0,3],[4,2,5,1]];

    var state = { k: 2, s: 2, tipo: 'max' };
    var visited = {};

    var slEl = root.querySelector('#ep0902_sl');
    var vlEl = root.querySelector('#ep0902_vl');
    var gridEl = root.querySelector('#ep0902_grid');
    var outEl = root.querySelector('#ep0902_out');
    var dbg = root.querySelector('#ep0902_debug');
    var formulaEl = root.querySelector('#ep0902_formula');
    var resetBtn = root.querySelector('#ep0902_reset');
    var kBtnsEl = root.querySelector('#ep0902_k_btns');
    var sBtnsEl = root.querySelector('#ep0902_s_btns');
    var btnMax = root.querySelector('#ep0902_max');
    var btnAvg = root.querySelector('#ep0902_avg');

    function btnStyle(active){
      return 'font-family:monospace;font-size:12px;font-weight:bold;width:34px;height:30px;border-radius:8px;cursor:pointer;' +
        (active ? 'background:#2980b9;color:white;border:1px solid #2471a3;' : 'background:#f3f4f6;color:#555;border:1px solid #ddd;');
    }

    function buildButtons(container, values, current, onPick){
      container.innerHTML = '';
      values.forEach(function(val){
        var b = document.createElement('button');
        b.textContent = val;
        b.style.cssText = btnStyle(val === current);
        b.addEventListener('click', function(){ onPick(val); });
        container.appendChild(b);
      });
    }

    function estiloTipoBotoes(){
      btnMax.style.background = state.tipo==='max' ? '#fff3cd' : '#f3f4f6';
      btnMax.style.borderColor = state.tipo==='max' ? '#f0ad4e' : '#ddd';
      btnMax.style.color = state.tipo==='max' ? '#7a5c00' : '#555';
      btnAvg.style.background = state.tipo==='avg' ? '#fff3cd' : '#f3f4f6';
      btnAvg.style.borderColor = state.tipo==='avg' ? '#f0ad4e' : '#ddd';
      btnAvg.style.color = state.tipo==='avg' ? '#7a5c00' : '#555';
    }

    function computeAll(k, s, tipo){
      var Oh = Math.floor((H - k)/s) + 1;
      var Ow = Math.floor((W - k)/s) + 1;
      var vals = [];
      for(var i=0; i<Oh; i++){
        vals.push([]);
        for(var j=0; j<Ow; j++){
          var janela = [];
          for(var r=i*s; r<i*s+k; r++) for(var c=j*s; c<j*s+k; c++) janela.push(X[r][c]);
          var resultado = tipo === 'max'
            ? Math.max.apply(null, janela)
            : janela.reduce(function(a,b){return a+b;},0)/janela.length;
          vals[i].push({ janela: janela, resultado: resultado });
        }
      }
      return { Oh: Oh, Ow: Ow, vals: vals };
    }

    var model = null;

    function rebuildModel(resetPosition){
      model = computeAll(state.k, state.s, state.tipo);
      outEl.style.gridTemplateColumns = 'repeat(' + model.Ow + ', 48px)';
      var maxPos = model.Oh*model.Ow - 1;
      slEl.max = maxPos;
      if(resetPosition){ slEl.value = 0; visited = {}; }
      else if(parseInt(slEl.value) > maxPos){ slEl.value = maxPos; }
      formulaEl.textContent = 'O_h = ⌊(' + H + ' − ' + state.k + ')/' + state.s + '⌋ + 1 = ' + model.Oh +
        '    O_w = ⌊(' + W + ' − ' + state.k + ')/' + state.s + '⌋ + 1 = ' + model.Ow;
      estiloTipoBotoes();
      render();
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/model.Ow), j = pos % model.Ow;
      var key = i+','+j;
      visited[key] = true;
      vlEl.textContent = '('+i+','+j+')';

      var winRowStart = i*state.s, winColStart = j*state.s;
      var alcancavel = []; // marca quais células de X são alcançadas por ALGUMA janela válida
      for(var r=0;r<H;r++){ alcancavel.push(new Array(W).fill(false)); }
      for(var oi=0; oi<model.Oh; oi++){
        for(var oj=0; oj<model.Ow; oj++){
          for(var r=oi*state.s; r<oi*state.s+state.k; r++)
            for(var c=oj*state.s; c<oj*state.s+state.k; c++)
              alcancavel[r][c] = true;
        }
      }

      gridEl.innerHTML = '';
      for(var r=0; r<H; r++){
        for(var c=0; c<W; c++){
          var dentroJanela = (r>=winRowStart && r<winRowStart+state.k && c>=winColStart && c<winColStart+state.k);
          var d = document.createElement('div');
          var base = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:13px;';
          if(dentroJanela){
            base += 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;';
          } else if(!alcancavel[r][c]){
            base += 'background:#eee;border:1px dashed #bbb;color:#999;';
          } else {
            base += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;';
          }
          d.style.cssText = base;
          d.textContent = X[r][c];
          gridEl.appendChild(d);
        }
      }

      var cur = model.vals[i][j];

      outEl.innerHTML = '';
      for(var oi2=0; oi2<model.Oh; oi2++){
        for(var oj2=0; oj2<model.Ow; oj2++){
          var isCurrent = (oi2===i && oj2===j);
          var wasVisited = !!visited[oi2+','+oj2];
          var od = document.createElement('div');
          var style = 'width:48px;height:48px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:11px;font-weight:bold;';
          if(isCurrent){
            style += 'background:#fff3cd;border:2px solid #f0ad4e;color:#7a5c00;';
          } else if(wasVisited){
            style += 'background:#e8f5e9;border:1px solid #a5d6a7;color:#2e7d32;';
          } else {
            style += 'background:#f3f4f6;border:1px dashed #ccc;color:#bbb;';
          }
          od.style.cssText = style;
          od.textContent = wasVisited ? model.vals[oi2][oj2].resultado.toFixed(2) : '?';
          outEl.appendChild(od);
        }
      }

      var totalVisitadas = Object.keys(visited).length;
      var totalPos = model.Oh*model.Ow;
      dbg.textContent = 'janela=['+cur.janela.join(', ')+']  |  tipo='+state.tipo+'  →  resultado='+cur.resultado.toFixed(4)+'   ['+totalVisitadas+'/'+totalPos+' posições exploradas]';
    }

    function setK(val){
      state.k = val;
      buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
      buildButtons(sBtnsEl, [1,2,3], state.s, setS);
      rebuildModel(true);
    }
    function setS(val){
      state.s = val;
      buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
      buildButtons(sBtnsEl, [1,2,3], state.s, setS);
      rebuildModel(true);
    }
    buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
    buildButtons(sBtnsEl, [1,2,3], state.s, setS);

    btnMax.addEventListener('click', function(){ state.tipo='max'; rebuildModel(true); });
    btnAvg.addEventListener('click', function(){ state.tipo='avg'; rebuildModel(true); });

    resetBtn.addEventListener('click', function(){
      visited = {};
      slEl.value = 0;
      render();
    });
    slEl.addEventListener('input', render);

    rebuildModel(true);
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0902');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.2:** Simulador EP09_02: Pooling Manual (máximo vs. média, con ventana k y stride s ajustables)


<figure id="fig-09-sim-ep0902">
  <img src="imagens/fig-09-sim-ep0902.png" alt=" Simulador EP09_02: Pooling Manual (máximo vs. média, con ventana k y stride s ajustables) " style="max-width:80%" />
  <figcaption><strong>Figura 9.2:</strong>  Simulador EP09_02: Pooling Manual (máximo vs. média, con ventana k y stride s ajustables) </figcaption>
</figure>

In [ ]:
%%writefile EP09_02.py
# Código Python

In [ ]:
TestSuite("EP09_02.py").run()

### EP09_03 🟡 Conteo de Parámetros Entrenables de una CNN

Este EP formaliza el conteo de parámetros entrenables de una *CNN*. Dada la descripción textual de una pequeña arquitectura, compuesta por capas convolucionales, de *pooling* y totalmente conectadas, determine, para cada capa, el número de parámetros entrenables y el total de la red.

La arquitectura debe interpretarse **secuencialmente**: la salida de una capa convolucional se convierte en la entrada de la siguiente capa compatible. Así, el número de canales producidos por una capa `CONV` determina el número de canales de entrada (`cin`) de la capa convolucional siguiente.

En una capa convolucional, es importante distinguir **canales de entrada** y **canales de salida**:

* $c_{in}$ (*channels in*) es el número de **canales que entran en la capa**. Una imagen en tonos de gris posee $c_{in}=1$, mientras que una imagen RGB posee $c_{in}=3$. En una capa convolucional intermedia, `cin` normalmente es igual al número de canales producidos por la capa `CONV` anterior.
* $c_{out}$ (*channels out*) es el número de **canales producidos por la capa**. Es igual al número de filtros utilizados. Por lo tanto, si una capa posee 16 filtros, produce $c_{out}=16$ canales.

Por ejemplo, considere la secuencia:

```text
CONV 3 3 1 8 1
POOL
CONV 3 3 8 16 1
POOL
FC 784 10 1
```

La primera convolución recibe una imagen con un canal y produce 8 canales. Después del *pooling*, la segunda convolución recibe esos 8 canales y produce 16 canales. La capa `POOL` no altera el número de canales, solo puede reducir las dimensiones espaciales. La capa `FC` recibe la cantidad de entradas informada en la propia descripción.

Cada filtro convolucional posee dimensiones

$$
k_h \times k_w \times c_{in}.
$$

Así, una capa con $c_{out}$ filtros posee

$$
k_h \cdot k_w \cdot c_{in} \cdot c_{out}
$$

pesos. Si hay sesgo, se añade un parámetro para cada filtro, totalizando más $c_{out}$ parámetros.

El punto central de este ejercicio es observar que la cantidad de parámetros de una capa convolucional **no depende de las dimensiones espaciales** ($H \times W$) del mapa de características. Esto ocurre debido al **comparticionamiento de pesos**: el mismo filtro se reutiliza en diferentes posiciones de la entrada.

#### 📋 Directrices de Implementación

1. **Entrada:** Leer el entero $L$ (número de capas de la arquitectura, en el orden en que se aplican).

2. **Capas:** Leer $L$ líneas, cada una describiendo una capa en uno de los tres formatos:

   * `CONV kh kw cin cout bias` — capa convolucional con *kernel* $k_h \times k_w$, $c_{in}$ canales de entrada, $c_{out}$ canales de salida y `bias` (0 o 1), indicando si hay sesgo por filtro;
   * `POOL` — capa de *pooling* (máximo o medio), que no posee parámetros entrenables y preserva el número de canales;
   * `FC in out bias` — capa totalmente conectada con `in` entradas, `out` salidas y `bias` (0 o 1), indicando si hay sesgo por neurona.

3. **Consistencia entre capas `CONV`:** en una secuencia de capas convolucionales, el `cin` de una capa debe corresponder al `cout` de la capa convolucional anterior. Una capa `POOL` no altera ese número de canales.

   Por ejemplo:

   ```text
   CONV 3 3 1 8 1
   POOL
   CONV 3 3 8 16 1
   ```

   La primera `CONV` produce 8 canales, que son recibidos por la segunda `CONV`. Por lo tanto, en la segunda capa, `cin=8` y `cout=16`.

4. **Parámetros de una capa `CONV`:**

   Cada uno de los $c_{out}$ filtros posee $k_h \cdot k_w \cdot c_{in}$ pesos. Por lo tanto,

   $$
   P_{\mathrm{CONV}} =
   k_h \cdot k_w \cdot c_{in} \cdot c_{out}
   +
   c_{out}\cdot\text{bias}.
   $$

5. **Parámetros de una capa `FC`:**

   $$
   P_{\mathrm{FC}} = 
   \text{in}\cdot\text{out}
   +
   \text{out}\cdot\text{bias}.
   $$

6. **Parámetros de una capa `POOL`:** siempre $0$.

7. **Total de la red:** sumar los parámetros entrenables de todas las capas.

8. **Salida:** Para cada capa, en el orden de lectura, imprimir `Camada i: P`, donde $i$ comienza en $1$ y $P$ es el número de parámetros de esa capa. Al final, imprimir `Total: T`.

#### 📐 Ejemplo para entender `cin` y `cout`

Considere la secuencia:

```text
CONV 3 3 1 8 1
POOL
CONV 3 3 8 16 1
```

En la primera capa:

* `cin=1`: entra un canal;
* `cout=8`: existen 8 filtros y, por lo tanto, salen 8 canales.

Cada filtro posee

$$
3\cdot3\cdot1=9
$$

pesos. Como existen 8 filtros:

$$
9\cdot8=72
$$

pesos. Con un sesgo por filtro:

$$
72+8=80.
$$

En la segunda capa:

* `cin=8`: entran los 8 canales producidos por la primera `CONV`;
* `cout=16`: existen 16 filtros y, por lo tanto, salen 16 canales.

Cada filtro posee

$$
3\cdot3\cdot8=72
$$

pesos. Como existen 16 filtros:

$$
72\cdot16=1152
$$

pesos. Con 16 sesgos:

$$
1152+16=1168.
$$

Así, las dos capas poseen, respectivamente, **80** y **1168 parámetros entrenables**.

Observe que `cout` **no es** $cin$ multiplicado por el número de filtros. El número de filtros es exactamente `cout`: cada filtro combina todos los canales de entrada y produce **un único canal de salida**.

#### 📌 Restricciones Computacionales

* **Independencia de la dimensión espacial:** la entrada no informa $H \times W$. El conteo de una capa `CONV` depende solo de `kh`, `kw`, `cin` y `cout`.
* **Consistencia de los canales:** para dos capas `CONV` consecutivas, el `cin` de la segunda debe ser igual al `cout` de la primera. Una capa `POOL` preserva el número de canales.
* **`bias` siempre 0 o 1:** multiplique directamente el término de sesgo por ese valor.
* **Capas `POOL` sin argumentos adicionales:** la línea contiene solo la palabra `POOL`.
* **Capas `FC`:** el número de entradas `in` se proporciona explícitamente. No es necesario calcular las dimensiones espaciales producidas por las capas anteriores.
* Todos los valores numéricos de entrada son enteros no negativos.

#### 🧠 Fundamentación Teórica

| Elemento                  | Papel en el conteo de parámetros                                                                                |
| ------------------------- | ---------------------------------------------------------------------------------------------------------------- |
| $c_{in}$                  | Número de canales recibidos por la capa                                                                           |
| $c_{out}$                 | Número de filtros y, por lo tanto, de canales producidos por la capa                                              |
| Filtro convolucional      | Cada filtro posee $k_h \cdot k_w \cdot c_{in}$ pesos y produce un canal de salida                                 |
| Compartición de pesos     | El mismo filtro se reutiliza en diferentes posiciones de la entrada, haciendo el conteo independiente de $H \times W$ |
| Sesgo                     | Un único parámetro adicional por filtro (`CONV`) o por neurona (`FC`)                                             |
| *Pooling*                 | Puede alterar $H \times W$, pero no posee parámetros entrenables y preserva el número de canales                  |
| Capa `FC`                 | Posee un peso para cada combinación entre entrada y neurona de salida                                             |

#### 🧩 Métodos del `morph.py` que pueden ayudar

Este ejercicio es puramente aritmético y no utiliza directamente funciones del `morph.py`. El conteo puede, sin embargo, verificarse en una arquitectura real implementada en PyTorch mediante:

```python
sum(p.numel() for p in modelo.parameters())
```

Esta expresión contabiliza los parámetros del modelo, incluyendo pesos y sesgos.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Próximas $L$ líneas: descripción de cada capa, en el formato `CONV kh kw cin cout bias`, `POOL` o `FC in out bias`.

**Salida:**

* $L$ líneas en el formato `Camada i: P`.
* Última línea: `Total: T`.

#### 📌 Ejemplos

| Entrada                                                                             | Salida                                                                                                          | Observación                                                                                                         |
| ----------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------ |
| 3<br>CONV 3 3 1 8 1<br>POOL<br>FC 1352 10 1                                         | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 13530<br>Total: 13610                                                 | Red simple con una convolución, *pooling* y capa de clasificación.                                                  |
| 5<br>CONV 3 3 1 8 1<br>POOL<br>CONV 3 3 8 16 1<br>POOL<br>FC 400 10 1               | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 1168<br>Camada 4: 0<br>Camada 5: 4010<br>Total: 5258                  | Pequeña CNN con dos convoluciones, dos *poolings* y una capa totalmente conectada.                                 |
| 6<br>CONV 3 3 1 8 1<br>POOL<br>CONV 3 3 8 16 1<br>POOL<br>FC 256 32 1<br>FC 32 10 1 | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 1168<br>Camada 4: 0<br>Camada 5: 8224<br>Camada 6: 330<br>Total: 9802 | CNN pequeña con dos convoluciones, *pooling* intermedio y dos capas totalmente conectadas para clasificación.       |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0903" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Conteo de Parámetros</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 compartilhamento de pesos</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>CONV</b> Bloque azul
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:50%;display:inline-block;"></span>
        <b>POOL</b> Cilindro verde
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;display:inline-block;transform:rotate(45deg);"></span>
        <b>FC</b> Rombo naranja
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;display:inline-block;"></span>
        <b>BATCH</b> Pila roja
      </span>
      <span style="display:flex;align-items:center;gap:3px;color:#666;">
        🖱️ Arrastra para mover capas
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">H×W</label>
            <span id="ep0903_hw_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">32×32</span>
          </div>
          <input id="ep0903_hw" style="width:100%;accent-color:#2980b9;height:4px;" max="64" min="8" step="2" type="range" value="32">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Canales</label>
            <span id="ep0903_cin_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">1</span>
          </div>
          <input id="ep0903_cin" style="width:100%;accent-color:#2980b9;height:4px;" max="3" min="1" step="1" type="range" value="1">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Batch</label>
            <span id="ep0903_batch_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">4</span>
          </div>
          <input id="ep0903_batch" style="width:100%;accent-color:#2980b9;height:4px;" max="16" min="1" step="1" type="range" value="4">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Capas</label>
            <span id="ep0903_nlayers_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">3</span>
          </div>
          <input id="ep0903_nlayers" style="width:100%;accent-color:#2980b9;height:4px;" max="6" min="1" step="1" type="range" value="3">
        </div>
      </div>
      
      <!-- Configuração das camadas compacta -->
      <div id="ep0903_layers_config" style="margin-bottom:8px;display:flex;flex-wrap:wrap;gap:6px;">
        <!-- Gerado dinamicamente -->
      </div>
      
      <div style="display:flex;gap:12px;align-items:center;font-size:10px;">
        <label style="display:flex;align-items:center;gap:4px;cursor:pointer;">
          <input id="ep0903_bias" type="checkbox" checked style="accent-color:#2980b9;width:14px;height:14px;">
          <span style="font-weight:bold;color:#2980b9;">Usar bias</span>
        </label>
      </div>
    </div>
    
    <!-- Visualização 3D -->
    <div style="position:relative;background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;margin-bottom:12px;min-height:400px;">
      <div style="position:absolute;top:8px;left:10px;color:white;font-weight:bold;font-size:14px;text-shadow:2px 2px 4px rgba(0,0,0,0.5);z-index:10;">
        🧠 Visualización 3D
      </div>
      
      <div style="position:absolute;top:8px;right:8px;display:flex;gap:4px;z-index:10;">
        <button id="ep0903_pause_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          ⏸️ Pausar
        </button>
        <button id="ep0903_reset_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          🔄 Reiniciar
        </button>
        <button id="ep0903_auto_layout_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          📐 Auto
        </button>
      </div>
      
      <canvas id="ep0903_canvas" style="width:100%;height:340px;display:block;cursor:grab;"></canvas>
      
      <div style="position:absolute;bottom:6px;right:8px;color:white;font-size:9px;background:rgba(0,0,0,0.5);padding:3px 8px;border-radius:14px;">
        🖱️ Arrastra capas | Scroll zoom | P pausa
      </div>
    </div>
    
    <!-- Resumo compacto -->
    <div id="ep0903_summary" style="background:#e3f2fd;border-radius:6px;padding:8px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;line-height:1.5;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var hwEl = root.querySelector('#ep0903_hw'), hwvEl = root.querySelector('#ep0903_hw_v');
    var cinEl = root.querySelector('#ep0903_cin'), cinvEl = root.querySelector('#ep0903_cin_v');
    var batchEl = root.querySelector('#ep0903_batch'), batchvEl = root.querySelector('#ep0903_batch_v');
    var nlayersEl = root.querySelector('#ep0903_nlayers'), nlayersvEl = root.querySelector('#ep0903_nlayers_v');
    var layersConfigEl = root.querySelector('#ep0903_layers_config');
    var biasEl = root.querySelector('#ep0903_bias');
    var summaryEl = root.querySelector('#ep0903_summary');
    var canvas = root.querySelector('#ep0903_canvas');
    var ctx = canvas.getContext('2d');
    var pauseBtn = root.querySelector('#ep0903_pause_btn');
    var resetBtn = root.querySelector('#ep0903_reset_btn');
    var autoLayoutBtn = root.querySelector('#ep0903_auto_layout_btn');
    
    // Estado da visualização
    var rotationX = -0.3;
    var rotationY = 0.5;
    var zoom = 1;
    var isDragging = false;
    var isDraggingLayer = false;
    var selectedLayer = null;
    var lastX = 0;
    var lastY = 0;
    var autoRotate = true;
    var isPaused = false;
    var lastInteractionTime = Date.now();
    var animationId = null;
    var time = 0;
    
    // Posições das camadas
    var layerPositions = [];
    var batchPosition = { x: -6, y: -0.5, z: 0 };
    
    function resizeCanvas() {
      canvas.width = canvas.clientWidth;
      canvas.height = canvas.clientHeight;
    }
    resizeCanvas();
    window.addEventListener('resize', resizeCanvas);
    
    function autoLayout() {
      var nlayers = parseInt(nlayersEl.value);
      var spacing = 4;
      var startX = -((nlayers) * spacing) / 2;
      
      batchPosition = { x: startX - spacing / 2, y: -0.5, z: 0 };
      
      layerPositions = [];
      for (var i = 0; i < nlayers; i++) {
        layerPositions.push({
          x: startX + (i + 0.5) * spacing,
          y: i * 1.5,
          z: 0
        });
      }
    }
    
    function togglePause() {
      isPaused = !isPaused;
      if (isPaused) {
        pauseBtn.textContent = '▶️';
        pauseBtn.style.background = 'rgba(76, 175, 80, 0.4)';
        autoRotate = false;
      } else {
        pauseBtn.textContent = '⏸️';
        pauseBtn.style.background = 'rgba(255,255,255,0.2)';
        autoRotate = true;
        lastInteractionTime = Date.now();
      }
    }
    
    function resetView() {
      rotationX = -0.3;
      rotationY = 0.5;
      zoom = 1;
      isPaused = false;
      autoRotate = true;
      pauseBtn.textContent = '⏸️';
      pauseBtn.style.background = 'rgba(255,255,255,0.2)';
      lastInteractionTime = Date.now();
      autoLayout();
    }
    
    pauseBtn.addEventListener('click', togglePause);
    resetBtn.addEventListener('click', resetView);
    autoLayoutBtn.addEventListener('click', autoLayout);
    
    document.addEventListener('keydown', function(e) {
      if (e.key === 'p' || e.key === 'P') togglePause();
      if (e.key === 'r' || e.key === 'R') resetView();
      if (e.key === 'a' || e.key === 'A') autoLayout();
    });
    
    function findLayerAt(mouseX, mouseY, layers) {
      var minDist = Infinity;
      var foundLayer = null;
      
      var batchProj = project(batchPosition);
      var batchDist = Math.sqrt(Math.pow(batchProj.x - mouseX, 2) + Math.pow(batchProj.y - mouseY, 2));
      if (batchDist < 50) {
        minDist = batchDist;
        foundLayer = { type: 'batch', index: -1 };
      }
      
      for (var i = 0; i < layerPositions.length && i < layers.length; i++) {
        var proj = project(layerPositions[i]);
        var dist = Math.sqrt(Math.pow(proj.x - mouseX, 2) + Math.pow(proj.y - mouseY, 2));
        
        if (dist < minDist && dist < 60) {
          minDist = dist;
          foundLayer = { type: 'layer', index: i };
        }
      }
      
      return foundLayer;
    }
    
    canvas.addEventListener('mousedown', function(e) {
      var rect = canvas.getBoundingClientRect();
      var mouseX = e.clientX - rect.left;
      var mouseY = e.clientY - rect.top;
      
      var layers = getCurrentLayersInfo();
      var clickedLayer = findLayerAt(mouseX, mouseY, layers);
      
      if (clickedLayer) {
        isDraggingLayer = true;
        selectedLayer = clickedLayer;
        canvas.style.cursor = 'grabbing';
      } else {
        isDragging = true;
        canvas.style.cursor = 'grabbing';
      }
      
      autoRotate = false;
      lastX = e.clientX;
      lastY = e.clientY;
      lastInteractionTime = Date.now();
    });
    
    canvas.addEventListener('mousemove', function(e) {
      var rect = canvas.getBoundingClientRect();
      var mouseX = e.clientX - rect.left;
      var mouseY = e.clientY - rect.top;
      
      if (isDraggingLayer && selectedLayer) {
        var deltaX = (e.clientX - lastX) * 0.05;
        var deltaY = -(e.clientY - lastY) * 0.05;
        
        if (selectedLayer.type === 'batch') {
          batchPosition.x += deltaX;
          batchPosition.y += deltaY;
        } else if (selectedLayer.type === 'layer') {
          layerPositions[selectedLayer.index].x += deltaX;
          layerPositions[selectedLayer.index].y += deltaY;
        }
        
        lastX = e.clientX;
        lastY = e.clientY;
      } else if (isDragging) {
        var deltaX = e.clientX - lastX;
        var deltaY = e.clientY - lastY;
        rotationY += deltaX * 0.01;
        rotationX += deltaY * 0.01;
        rotationX = Math.max(-1.5, Math.min(1.5, rotationX));
        lastX = e.clientX;
        lastY = e.clientY;
      }
      
      if (!isDragging && !isDraggingLayer) {
        var layers = getCurrentLayersInfo();
        var hoveredLayer = findLayerAt(mouseX, mouseY, layers);
        canvas.style.cursor = hoveredLayer ? 'pointer' : 'grab';
      }
    });
    
    canvas.addEventListener('mouseup', function() {
      isDragging = false;
      isDraggingLayer = false;
      selectedLayer = null;
      canvas.style.cursor = 'grab';
      if (!isPaused) {
        setTimeout(function() { autoRotate = true; }, 3000);
      }
    });
    
    canvas.addEventListener('mouseleave', function() {
      isDragging = false;
      isDraggingLayer = false;
      selectedLayer = null;
      canvas.style.cursor = 'grab';
    });
    
    canvas.addEventListener('wheel', function(e) {
      e.preventDefault();
      zoom *= (1 + e.deltaY * 0.001);
      zoom = Math.max(0.5, Math.min(2, zoom));
      lastInteractionTime = Date.now();
      autoRotate = false;
      if (!isPaused) {
        setTimeout(function() { autoRotate = true; }, 3000);
      }
    });
    
    function rotateX(point, angle) {
      var cos = Math.cos(angle);
      var sin = Math.sin(angle);
      return { x: point.x, y: point.y * cos - point.z * sin, z: point.y * sin + point.z * cos };
    }
    
    function rotateY(point, angle) {
      var cos = Math.cos(angle);
      var sin = Math.sin(angle);
      return { x: point.x * cos - point.z * sin, y: point.y, z: point.x * sin + point.z * cos };
    }
    
    function project(point) {
      var rotated = rotateX(point, rotationX);
      rotated = rotateY(rotated, rotationY);
      var scale = zoom * 30;
      return { x: canvas.width / 2 + rotated.x * scale, y: canvas.height / 2 - rotated.y * scale, z: rotated.z };
    }
    
    function shadeColor(color, percent) {
      var num = parseInt(color.replace('#', ''), 16);
      var amt = Math.round(2.55 * percent);
      var R = (num >> 16) + amt;
      var G = (num >> 8 & 0x00FF) + amt;
      var B = (num & 0x0000FF) + amt;
      return '#' + (0x1000000 + (R < 255 ? R < 1 ? 0 : R : 255) * 0x10000 + (G < 255 ? G < 1 ? 0 : G : 255) * 0x100 + (B < 255 ? B < 1 ? 0 : B : 255)).toString(16).slice(1);
    }
    
    function draw3DBox(x, y, z, width, height, depth, color, opacity, label, shape) {
      shape = shape || 'box';
      if (shape === 'cylinder') { draw3DCylinder(x, y, z, width, height, depth, color, opacity, label); return; }
      if (shape === 'diamond') { draw3DDiamond(x, y, z, width, height, depth, color, opacity, label); return; }
      
      var vertices = [
        {x: x - width/2, y: y - height/2, z: z - depth/2},
        {x: x + width/2, y: y - height/2, z: z - depth/2},
        {x: x + width/2, y: y + height/2, z: z - depth/2},
        {x: x - width/2, y: y + height/2, z: z - depth/2},
        {x: x - width/2, y: y - height/2, z: z + depth/2},
        {x: x + width/2, y: y - height/2, z: z + depth/2},
        {x: x + width/2, y: y + height/2, z: z + depth/2},
        {x: x - width/2, y: y + height/2, z: z + depth/2}
      ];
      
      var projected = vertices.map(function(v) { return project(v); });
      
      var faces = [
        {vertices: [0, 1, 2, 3], color: shadeColor(color, -20)},
        {vertices: [4, 5, 6, 7], color: shadeColor(color, 20)},
        {vertices: [0, 1, 5, 4], color: shadeColor(color, -40)},
        {vertices: [2, 3, 7, 6], color: shadeColor(color, 40)},
        {vertices: [1, 2, 6, 5], color: shadeColor(color, -10)},
        {vertices: [0, 3, 7, 4], color: shadeColor(color, 10)}
      ];
      
      faces.sort(function(a, b) {
        var za = a.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 4;
        var zb = b.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 4;
        return za - zb;
      });
      
      faces.forEach(function(face) {
        ctx.beginPath();
        ctx.moveTo(projected[face.vertices[0]].x, projected[face.vertices[0]].y);
        for (var i = 1; i < face.vertices.length; i++) {
          ctx.lineTo(projected[face.vertices[i]].x, projected[face.vertices[i]].y);
        }
        ctx.closePath();
        ctx.fillStyle = face.color;
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      });
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function draw3DCylinder(x, y, z, width, height, depth, color, opacity, label) {
      var segments = 12;
      var topVertices = [];
      var bottomVertices = [];
      
      for (var i = 0; i < segments; i++) {
        var angle = (i / segments) * Math.PI * 2;
        var cx = x + Math.cos(angle) * width / 2;
        var cz = z + Math.sin(angle) * depth / 2;
        topVertices.push({x: cx, y: y + height/2, z: cz});
        bottomVertices.push({x: cx, y: y - height/2, z: cz});
      }
      
      var projectedTop = topVertices.map(function(v) { return project(v); });
      var projectedBottom = bottomVertices.map(function(v) { return project(v); });
      
      for (var i = 0; i < segments; i++) {
        var next = (i + 1) % segments;
        ctx.beginPath();
        ctx.moveTo(projectedTop[i].x, projectedTop[i].y);
        ctx.lineTo(projectedTop[next].x, projectedTop[next].y);
        ctx.lineTo(projectedBottom[next].x, projectedBottom[next].y);
        ctx.lineTo(projectedBottom[i].x, projectedBottom[i].y);
        ctx.closePath();
        ctx.fillStyle = shadeColor(color, (i % 2 === 0) ? -10 : 10);
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      }
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function draw3DDiamond(x, y, z, width, height, depth, color, opacity, label) {
      var vertices = [
        {x: x, y: y + height/2, z: z},
        {x: x + width/2, y: y, z: z},
        {x: x, y: y, z: z + depth/2},
        {x: x - width/2, y: y, z: z},
        {x: x, y: y, z: z - depth/2},
        {x: x, y: y - height/2, z: z}
      ];
      
      var projected = vertices.map(function(v) { return project(v); });
      
      var faces = [
        {vertices: [0, 1, 2], color: shadeColor(color, -20)},
        {vertices: [0, 2, 3], color: shadeColor(color, 20)},
        {vertices: [0, 3, 4], color: shadeColor(color, -10)},
        {vertices: [0, 4, 1], color: shadeColor(color, 10)},
        {vertices: [5, 1, 2], color: shadeColor(color, -30)},
        {vertices: [5, 2, 3], color: shadeColor(color, 30)},
        {vertices: [5, 3, 4], color: shadeColor(color, -20)},
        {vertices: [5, 4, 1], color: shadeColor(color, 20)}
      ];
      
      faces.sort(function(a, b) {
        var za = a.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 3;
        var zb = b.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 3;
        return za - zb;
      });
      
      faces.forEach(function(face) {
        ctx.beginPath();
        ctx.moveTo(projected[face.vertices[0]].x, projected[face.vertices[0]].y);
        for (var i = 1; i < face.vertices.length; i++) {
          ctx.lineTo(projected[face.vertices[i]].x, projected[face.vertices[i]].y);
        }
        ctx.closePath();
        ctx.fillStyle = face.color;
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      });
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function drawImageBatch(x, y, z, width, height, numImages, color) {
      var imageDepth = 0.3;
      var gap = 0.1;
      var totalDepth = numImages * (imageDepth + gap);
      var startZ = z - totalDepth / 2;
      
      for (var i = 0; i < numImages; i++) {
        var imageZ = startZ + i * (imageDepth + gap);
        var alpha = 0.3 + (i / numImages) * 0.5;
        draw3DBox(x, y, imageZ, width, height, imageDepth, color, alpha, null, 'box');
      }
    }
    
    function drawConnection(x1, y1, z1, x2, y2, z2, animated) {
      var start = project({x: x1, y: y1, z: z1});
      var end = project({x: x2, y: y2, z: z2});
      var midX = (start.x + end.x) / 2;
      var midY = Math.min(start.y, end.y) - 20;
      
      if (animated && !isPaused) {
        var pulse = Math.sin(time * 0.002) * 0.5 + 0.5;
        ctx.strokeStyle = 'rgba(255, 255, 255, ' + (0.3 + pulse * 0.3) + ')';
        ctx.lineWidth = 1.5 + pulse;
      } else {
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1.5;
      }
      
      ctx.setLineDash([4, 4]);
      ctx.beginPath();
      ctx.moveTo(start.x, start.y);
      ctx.quadraticCurveTo(midX, midY, end.x, end.y);
      ctx.stroke();
      ctx.setLineDash([]);
    }
    
    function getCurrentLayersInfo() {
      var hw = parseInt(hwEl.value);
      var cin = parseInt(cinEl.value);
      var nlayers = parseInt(nlayersEl.value);
      var currentH = hw;
      var currentW = hw;
      var currentCin = cin;
      var currentFCInput = 0;
      var layersInfo = [];
      
      for (var i = 0; i < nlayers; i++) {
        var typeSelect = root.querySelector('#ep0903_type_' + i);
        var type = typeSelect ? typeSelect.value : 'CONV';
        var inputStr = '';
        var outputStr = '';
        
        if (type === 'CONV' || type === 'POOL') {
          inputStr = currentH + '×' + currentW + '×' + currentCin;
        } else if (type === 'FC') {
          // Se a camada anterior era FC, usa a saída dela
          if (i > 0 && layersInfo[i-1].type === 'FC') {
            inputStr = currentFCInput;
          } else {
            // Se veio de CONV/POOL, faz flatten
            inputStr = (currentH * currentW * currentCin);
            currentFCInput = inputStr;
          }
        }
        
        if (type === 'CONV') {
          var kh = parseInt(root.querySelector('#ep0903_kh_' + i).value);
          var kw = parseInt(root.querySelector('#ep0903_kw_' + i).value);
          var cout = parseInt(root.querySelector('#ep0903_cout_' + i).value);
          var outH = currentH - kh + 1;
          var outW = currentW - kw + 1;
          outputStr = outH + '×' + outW + '×' + cout;
          currentH = outH;
          currentW = outW;
          currentCin = cout;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'POOL') {
          var poolSize = parseInt(root.querySelector('#ep0903_pool_size_' + i).value);
          var poolOutH = Math.floor(currentH / poolSize);
          var poolOutW = Math.floor(currentW / poolSize);
          outputStr = poolOutH + '×' + poolOutW + '×' + currentCin;
          currentH = poolOutH;
          currentW = poolOutW;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'FC') {
          var fcout = parseInt(root.querySelector('#ep0903_fcout_' + i).value);
          outputStr = fcout;
          currentFCInput = fcout;
          // Após FC, não há mais dimensões espaciais
          currentH = 0;
          currentW = 0;
          currentCin = 0;
        }
        
        layersInfo.push({
          type: type,
          input: inputStr,
          output: outputStr,
          h: currentH,
          w: currentW,
          cin: currentCin
        });
      }
      
      return layersInfo;
    }
    
    function render3D(layers, batchSize) {
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      if (!isPaused) time += 16;
      if (autoRotate && !isDragging && !isPaused && Date.now() - lastInteractionTime > 3000) rotationY += 0.005;
      
      // Grid
      ctx.strokeStyle = 'rgba(255, 255, 255, 0.08)';
      ctx.lineWidth = 0.5;
      for (var i = -10; i <= 10; i++) {
        var start = project({x: i * 2, y: -2, z: -10 * 2});
        var end = project({x: i * 2, y: -2, z: 10 * 2});
        ctx.beginPath(); ctx.moveTo(start.x, start.y); ctx.lineTo(end.x, end.y); ctx.stroke();
        start = project({x: -10 * 2, y: -2, z: i * 2});
        end = project({x: 10 * 2, y: -2, z: i * 2});
        ctx.beginPath(); ctx.moveTo(start.x, start.y); ctx.lineTo(end.x, end.y); ctx.stroke();
      }
      
      var colors = ['#4a90e2', '#50e3c2', '#f5a623', '#d0021b', '#8b572a', '#9013fe'];
      
      // Batch (apenas se a primeira camada for CONV ou POOL)
      if (layers.length > 0 && (layers[0].type === 'CONV' || layers[0].type === 'POOL')) {
        var inputWidth = Math.max(1, Math.min(4, layers[0].h / 8));
        var inputHeight = Math.max(1, Math.min(4, layers[0].w / 8));
        var labelPos = project({x: batchPosition.x, y: batchPosition.y + inputHeight/2 + 0.7, z: batchPosition.z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 9px Arial';
        ctx.textAlign = 'center';
        ctx.fillText('BATCH: ' + batchSize, labelPos.x, labelPos.y);
        drawImageBatch(batchPosition.x, batchPosition.y, batchPosition.z, inputWidth, inputHeight, batchSize, '#ff6b6b');
        
        // Conexão batch -> primeira camada
        drawConnection(batchPosition.x + inputWidth/2, batchPosition.y, batchPosition.z, layerPositions[0].x - Math.max(1, Math.min(4, layers[0].h / 8))/2, layerPositions[0].y, layerPositions[0].z, true);
      }
      
      // Conexões entre camadas
      for (var i = 0; i < layers.length - 1 && i < layerPositions.length - 1; i++) {
        drawConnection(layerPositions[i].x + 1, layerPositions[i].y, layerPositions[i].z, layerPositions[i + 1].x - 1, layerPositions[i + 1].y, layerPositions[i + 1].z, true);
      }
      
      // Camadas
      for (var i = 0; i < layers.length && i < layerPositions.length; i++) {
        var layer = layers[i];
        var pos = layerPositions[i];
        
        var color = colors[i % colors.length];
        var label = '';
        
        if (layer.type === 'CONV') {
          var visWidth = Math.max(1, Math.min(4, layer.h / 8));
          var visHeight = Math.max(1, Math.min(4, layer.w / 8));
          var visDepth = Math.max(0.3, Math.min(2, layer.cin / 4));
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, visWidth, visHeight, visDepth, color, 0.7, label, 'box');
        } else if (layer.type === 'POOL') {
          var visWidth = Math.max(1, Math.min(4, layer.h / 8));
          var visHeight = Math.max(1, Math.min(4, layer.w / 8));
          var visDepth = Math.max(0.3, Math.min(2, layer.cin / 4));
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, visWidth, visHeight, visDepth, color, 0.7, label, 'cylinder');
        } else if (layer.type === 'FC') {
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, 1, 1, 1, color, 0.7, label, 'diamond');
        }
      }
      
      animationId = requestAnimationFrame(function() { render3D(layers, batchSize); });
    }
    
    function generateLayerConfig() {
      var nlayers = parseInt(nlayersEl.value);
      var html = '';
      
      for (var i = 0; i < nlayers; i++) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:6px 8px;display:flex;gap:6px;align-items:center;flex-wrap:wrap;font-size:9px;">';
        html += '<span style="font-weight:bold;color:#666;">' + (i+1) + ':</span>';
        html += '<select id="ep0903_type_' + i + '" style="padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;">';
        html += '<option value="CONV"' + (i < 2 ? ' selected' : '') + '>CONV</option>';
        html += '<option value="POOL">POOL</option>';
        html += '<option value="FC"' + (i >= 2 ? ' selected' : '') + '>FC</option>';
        html += '</select>';
        html += '<div id="ep0903_params_' + i + '" style="display:flex;gap:3px;flex-wrap:wrap;"></div>';
        html += '</div>';
      }
      
      layersConfigEl.innerHTML = html;
      
      for (var i = 0; i < nlayers; i++) {
        (function(index) {
          var typeSelect = root.querySelector('#ep0903_type_' + index);
          typeSelect.addEventListener('change', function() {
            updateLayerParams(index);
            render();
          });
          updateLayerParams(index);
        })(i);
      }
      
      autoLayout();
    }
    
    function updateLayerParams(index) {
      var typeSelect = root.querySelector('#ep0903_type_' + index);
      var paramsDiv = root.querySelector('#ep0903_params_' + index);
      var type = typeSelect.value;
      
      if (type === 'CONV') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_kh_' + index + '" value="3" min="1" max="7" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="kernel h">' +
          '<span style="font-size:8px;">×</span>' +
          '<input type="number" id="ep0903_kw_' + index + '" value="3" min="1" max="7" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="kernel w">' +
          '<input type="number" id="ep0903_cout_' + index + '" value="' + (index === 0 ? '8' : '16') + '" min="1" max="64" style="width:36px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="filtros">';
      } else if (type === 'POOL') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_pool_size_' + index + '" value="2" min="2" max="4" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="tamaño del pool">';
      } else if (type === 'FC') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_fcout_' + index + '" value="10" min="1" max="100" style="width:36px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="salidas">';
      }
      
      var inputs = paramsDiv.querySelectorAll('input');
      inputs.forEach(function(input) {
        input.addEventListener('input', render);
        input.addEventListener('change', render);
      });
    }
    
    function render() {
      var hw = parseInt(hwEl.value);
      var cin = parseInt(cinEl.value);
      var batchSize = parseInt(batchEl.value);
      var nlayers = parseInt(nlayersEl.value);
      var bias = biasEl.checked ? 1 : 0;
      
      hwvEl.textContent = hw + '×' + hw;
      cinvEl.textContent = cin;
      batchvEl.textContent = batchSize;
      nlayersvEl.textContent = nlayers;
      
      var currentH = hw;
      var currentW = hw;
      var currentCin = cin;
      var currentFCInput = 0;
      var totalParams = 0;
      var layersInfo = [];
      var summaryHTML = '';
      
      for (var i = 0; i < nlayers; i++) {
        var typeSelect = root.querySelector('#ep0903_type_' + i);
        var type = typeSelect.value;
        var params = 0;
        var inputStr = '';
        var outputStr = '';
        
        // Determinar entrada
        if (type === 'CONV' || type === 'POOL') {
          inputStr = currentH + '×' + currentW + '×' + currentCin;
        } else if (type === 'FC') {
          if (i > 0 && layersInfo[i-1].type === 'FC') {
            inputStr = currentFCInput;
          } else {
            inputStr = (currentH * currentW * currentCin);
            currentFCInput = inputStr;
          }
        }
        
        // Processar camada
        if (type === 'CONV') {
          var kh = parseInt(root.querySelector('#ep0903_kh_' + i).value);
          var kw = parseInt(root.querySelector('#ep0903_kw_' + i).value);
          var cout = parseInt(root.querySelector('#ep0903_cout_' + i).value);
          var outH = currentH - kh + 1;
          var outW = currentW - kw + 1;
          params = kh * kw * currentCin * cout + cout * bias;
          outputStr = outH + '×' + outW + '×' + cout;
          currentH = outH;
          currentW = outW;
          currentCin = cout;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'POOL') {
          var poolSize = parseInt(root.querySelector('#ep0903_pool_size_' + i).value);
          var poolOutH = Math.floor(currentH / poolSize);
          var poolOutW = Math.floor(currentW / poolSize);
          params = 0;
          outputStr = poolOutH + '×' + poolOutW + '×' + currentCin;
          currentH = poolOutH;
          currentW = poolOutW;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'FC') {
          var fcout = parseInt(root.querySelector('#ep0903_fcout_' + i).value);
          var fcin = parseInt(inputStr);
          params = fcin * fcout + fcout * bias;
          outputStr = fcout;
          currentFCInput = fcout;
          currentH = 0;
          currentW = 0;
          currentCin = 0;
        }
        
        totalParams += params;
        layersInfo.push({
          type: type,
          input: inputStr,
          output: outputStr,
          params: params,
          h: currentH,
          w: currentW,
          cin: currentCin
        });
        
        summaryHTML += '<span style="color:' + (type === 'CONV' ? '#2980b9' : type === 'POOL' ? '#666' : '#009933') + ';font-weight:bold;">' + (i+1) + ' (' + type + '):</span> ';
        summaryHTML += inputStr + ' → ' + outputStr;
        summaryHTML += ' [' + params.toLocaleString('pt-BR') + ']<br>';
      }
      
      summaryHTML += '<b>Total: ' + totalParams.toLocaleString('pt-BR') + ' parâmetros</b>';
      summaryEl.innerHTML = summaryHTML;
      
      if (animationId) cancelAnimationFrame(animationId);
      render3D(layersInfo, batchSize);
    }
    
    // Inicializar
    autoLayout();
    generateLayerConfig();
    render();
    
    // Event listeners
    hwEl.addEventListener('input', render);
    cinEl.addEventListener('input', render);
    batchEl.addEventListener('input', render);
    nlayersEl.addEventListener('input', function() { generateLayerConfig(); render(); });
    biasEl.addEventListener('change', render);
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0903');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.3:** Simulador EP09_03: Conteo de Parámetros — Convolución vs. Capa Totalmente Conectada


<figure id="fig-09-sim-ep0903">
  <img src="imagens/fig-09-sim-ep0903.png" alt=" Simulador EP09_03: Conteo de Parámetros — Convolución vs. Capa Totalmente Conectada " style="max-width:80%" />
  <figcaption><strong>Figura 9.3:</strong>  Simulador EP09_03: Conteo de Parámetros — Convolución vs. Capa Totalmente Conectada </figcaption>
</figure>

In [ ]:
%%writefile EP09_03.py
# Código Python

In [ ]:
TestSuite("EP09_03.py").run()

### EP09_04 🟡 Intersección sobre Unión (IoU) y Supresión de No-Máximos (NMS)

Los modelos de detección de objetos pueden producir **varias cajas delimitadoras candidatas** para un mismo objeto, con diferentes posiciones y puntuaciones de confianza. La etapa de posprocesamiento responsable de eliminar esas detecciones redundantes es la **Supresión de No-Máximos (NMS)**, cuya operación fundamental utiliza la métrica de **Intersección sobre Unión (IoU)**.

La NMS utiliza esta medida para decidir qué cajas deben mantenerse. En general, la caja con mayor confianza se selecciona primero; a continuación, las cajas que presentan una IoU por encima de un determinado umbral con la caja seleccionada se consideran redundantes y se eliminan. El proceso se repite hasta que no queden cajas candidatas.

En este ejercicio, deberás implementar el algoritmo de NMS desde cero, calculando la IoU entre cajas y aplicando sucesivamente el criterio de selección y supresión para producir el conjunto final de detecciones.

#### 📋 Directrices de Implementación

1. **Entrada:** Leer el entero $N$ (número de cajas candidatas) y el umbral real $\tau$ (umbral de IoU para la supresión), en la misma línea.

2. **Cajas:** Leer $N$ líneas, cada una con cinco valores reales:

   `x1 y1 x2 y2 score`

   donde $(x_1,y_1)$ representa la esquina superior izquierda, $(x_2,y_2)$ la esquina inferior derecha y `score` la puntuación de confianza.

3. **Intersección sobre Unión:** Para dos cajas $A$ y $B$,

   $$
   IoU(A,B)=
   \frac{\operatorname{Área}(A\cap B)}
   {\operatorname{Área}(A\cup B)}.
   $$

   El área de intersección debe calcularse a partir de la superposición de los intervalos en $x$ e $y$. Si no hay superposición, el área de intersección es cero.

4. **Algoritmo voraz de NMS:**

   a. Ordena las cajas por `score` descendente. En caso de empate, mantén el orden original de lectura.

   b. Selecciona la caja de mayor puntuación entre las cajas restantes y añádela al conjunto de salida.

   c. Calcula la IoU entre la caja seleccionada y **todas las cajas aún restantes**. Suprime las cajas para las cuales

   $$
   \text{IoU} > \tau.
   $$

   d. Repite los pasos (b) y (c) hasta que no queden cajas.

5. **Salida:** Para cada caja mantenida, en el orden en que fue seleccionada, imprimir su índice original (posición de lectura, comenzando en $0$) y su `score`, formateado con 4 decimales. Al final, imprimir:

   `Total mantenidas: X`

#### 📌 Restricciones Computacionales

* **Supresión estricta:** solo las cajas con $\text{IoU} > \tau$ se suprimen. Las cajas con $\text{IoU} = \tau$ se mantienen.
* **Índices originales:** la salida hace referencia a la posición en que cada caja fue leída en la entrada (comenzando en $0$), y no a su posición después de la ordenación.
* **Ordenación estable:** en caso de `score` iguales, debe preservarse el orden original de lectura.
* **Rectángulos alineados a los ejes:** todas las cajas se especifican mediante dos esquinas, con $x_1 < x_2$ e $y_1 < y_2$ garantizados en la entrada.
* **Coordenadas y puntuaciones:** los valores reales pueden ser positivos o negativos, según los límites definidos por la entrada, pero las dimensiones de las cajas son siempre positivas.

#### 🧠 Fundamentación Teórica

| Elemento                | Papel en el posprocesamiento de detección                                                                                              |
| ----------------------- | --------------------------------------------------------------------------------------------------------------------------------------- |
| IoU                     | Cuantifica la superposición espacial entre dos cajas; $\text{IoU}=1$ para cajas idénticas y $\text{IoU}=0$ para cajas sin superposición |
| Ordenación por confianza | Hace que la caja de mayor `score` se analice primero                                                                                    |
| Umbral $\tau$           | Define la cantidad de superposición necesaria para que una caja se considere redundante                                                 |
| Supresión               | Elimina cajas que presentan una gran superposición con una caja ya seleccionada                                                         |
| Cajas distantes         | Poseen IoU cercana a cero y, en general, no se suprimen por esta regla                                                                  |

#### 🧩 Métodos de `morph.py` que pueden ayudar

* `mm.IoU(boxA, boxB)` — calcula la métrica de IoU, pero espera las cajas en el formato $(x,y,w,h)$, es decir, esquina superior izquierda, ancho y alto. La entrada de este ejercicio utiliza el formato $(x_1,y_1,x_2,y_2)$. La conversión es directa:

  $$
  w=x_2-x_1,\qquad h=y_2-y_1.
  $$

  El uso de esta función es opcional. El objetivo principal del ejercicio es implementar correctamente el proceso de selección y supresión de la NMS.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: entero $N$ y real $\tau$.
* Siguientes $N$ líneas: $x_1\ y_1\ x_2\ y_2\ \text{score}$.

**Salida:**

* Una línea por caja mantenida, en el orden de selección: `índice score`.
* Última línea: `Total mantenidas: X`.

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0904" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: IoU y Supresión de No Máximos</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 NMS</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Caja seleccionada</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:2px;display:inline-block;"></span>
        <b>Caja mantenida</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Caja suprimida</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Caja candidata</b>
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Nº de cajas</label>
            <span id="ep0904_n_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">5</span>
          </div>
          <input id="ep0904_n" style="width:100%;accent-color:#2980b9;height:4px;" max="8" min="2" step="1" type="range" value="5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Umbral τ (IoU)</label>
            <span id="ep0904_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">0.50</span>
          </div>
          <input id="ep0904_tau" style="width:100%;accent-color:#2980b9;height:4px;" max="1.0" min="0.1" step="0.05" type="range" value="0.5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Ejemplo</label>
            <span id="ep0904_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Patrón</span>
          </div>
          <select id="ep0904_example" style="width:100%;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;">
            <option value="padrao">Ejemplo Patrón</option>
            <option value="agrupado">Cajas Agrupadas</option>
            <option value="disperso">Cajas Dispersas</option>
            <option value="aninhado">Cajas Anidadas</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;justify-content:center;">
          <button id="ep0904_run_btn" style="background:#2980b9;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;transition:all 0.3s;">
            ▶️ Ejecutar NMS
          </button>
        </div>
      </div>
      
      <!-- Caixas configuráveis -->
      <div id="ep0904_boxes_config" style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:8px;">
        <!-- Gerado dinamicamente -->
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas da visualização -->
      <div style="position:relative;background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;min-height:300px;">
        <div style="position:absolute;top:8px;left:10px;color:white;font-weight:bold;font-size:14px;text-shadow:2px 2px 4px rgba(0,0,0,0.5);z-index:10;">
          🎯 Visualización de las Cajas
        </div>
        <canvas id="ep0904_canvas" style="width:100%;height:280px;display:block;"></canvas>
      </div>
      
      <!-- Passo a passo -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:310px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📋 Paso a Paso de la NMS
        </div>
        <div id="ep0904_steps" style="font-family:monospace;font-size:10px;line-height:1.6;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resumo -->
    <div id="ep0904_summary" style="background:#e3f2fd;border-radius:6px;padding:8px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;line-height:1.5;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var nEl = root.querySelector('#ep0904_n'), nvEl = root.querySelector('#ep0904_n_v');
    var tauEl = root.querySelector('#ep0904_tau'), tauvEl = root.querySelector('#ep0904_tau_v');
    var exampleEl = root.querySelector('#ep0904_example');
    var boxesConfigEl = root.querySelector('#ep0904_boxes_config');
    var canvas = root.querySelector('#ep0904_canvas');
    var ctx = canvas.getContext('2d');
    var stepsEl = root.querySelector('#ep0904_steps');
    var summaryEl = root.querySelector('#ep0904_summary');
    var runBtn = root.querySelector('#ep0904_run_btn');
    
    // Estado
    var boxes = [];
    var selectedBoxes = [];
    var suppressedBoxes = [];
    
    // Exemplos pré-definidos
    var examples = {
      padrao: [
        {x1: 10, y1: 10, x2: 50, y2: 50, score: 0.9},
        {x1: 20, y1: 20, x2: 60, y2: 60, score: 0.8},
        {x1: 30, y1: 15, x2: 70, y2: 55, score: 0.7},
        {x1: 80, y1: 80, x2: 120, y2: 120, score: 0.6},
        {x1: 85, y1: 85, x2: 125, y2: 125, score: 0.5}
      ],
      agrupado: [
        {x1: 10, y1: 10, x2: 50, y2: 50, score: 0.9},
        {x1: 15, y1: 15, x2: 55, y2: 55, score: 0.85},
        {x1: 20, y1: 20, x2: 60, y2: 60, score: 0.8},
        {x1: 25, y1: 25, x2: 65, y2: 65, score: 0.75},
        {x1: 30, y1: 30, x2: 70, y2: 70, score: 0.7}
      ],
      disperso: [
        {x1: 10, y1: 10, x2: 40, y2: 40, score: 0.9},
        {x1: 80, y1: 10, x2: 110, y2: 40, score: 0.8},
        {x1: 10, y1: 80, x2: 40, y2: 110, score: 0.7},
        {x1: 80, y1: 80, x2: 110, y2: 110, score: 0.6},
        {x1: 45, y1: 45, x2: 75, y2: 75, score: 0.5}
      ],
      aninhado: [
        {x1: 10, y1: 10, x2: 90, y2: 90, score: 0.9},
        {x1: 20, y1: 20, x2: 80, y2: 80, score: 0.8},
        {x1: 30, y1: 30, x2: 70, y2: 70, score: 0.7},
        {x1: 40, y1: 40, x2: 60, y2: 60, score: 0.6},
        {x1: 45, y1: 45, x2: 55, y2: 55, score: 0.5}
      ]
    };
    
    // Ajustar tamanho do canvas
    function resizeCanvas() {
      canvas.width = canvas.clientWidth;
      canvas.height = canvas.clientHeight;
    }
    resizeCanvas();
    window.addEventListener('resize', function() {
      resizeCanvas();
      render();
    });
    
    // Carregar exemplo
    function loadExample(name) {
      boxes = JSON.parse(JSON.stringify(examples[name]));
      selectedBoxes = [];
      suppressedBoxes = [];
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Caixas carregadas: ' + boxes.length + '. Clique em "Executar NMS".';
    }
    
    // Gerar configuração das caixas
    function generateBoxesConfig() {
      var html = '';
      for (var i = 0; i < boxes.length; i++) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:4px 6px;font-size:9px;">';
        html += '<span style="font-weight:bold;color:#666;">#' + i + ':</span>';
        html += '<input type="number" id="ep0904_x1_' + i + '" value="' + boxes[i].x1 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="x1">';
        html += '<input type="number" id="ep0904_y1_' + i + '" value="' + boxes[i].y1 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="y1">';
        html += '<input type="number" id="ep0904_x2_' + i + '" value="' + boxes[i].x2 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="x2">';
        html += '<input type="number" id="ep0904_y2_' + i + '" value="' + boxes[i].y2 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="y2">';
        html += '<input type="number" id="ep0904_score_' + i + '" value="' + boxes[i].score + '" step="0.05" min="0" max="1" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="score">';
        html += '</div>';
      }
      boxesConfigEl.innerHTML = html;
      
      // Adicionar event listeners
      for (var i = 0; i < boxes.length; i++) {
        (function(index) {
          ['x1', 'y1', 'x2', 'y2', 'score'].forEach(function(field) {
            var input = root.querySelector('#ep0904_' + field + '_' + index);
            if (input) {
              input.addEventListener('input', function() {
                boxes[index][field] = parseFloat(input.value) || 0;
                selectedBoxes = [];
                suppressedBoxes = [];
                render();
              });
            }
          });
        })(i);
      }
    }
    
    // Calcular IoU
    function calculateIoU(box1, box2) {
      var x1 = Math.max(box1.x1, box2.x1);
      var y1 = Math.max(box1.y1, box2.y1);
      var x2 = Math.min(box1.x2, box2.x2);
      var y2 = Math.min(box1.y2, box2.y2);
      
      var intersectionArea = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
      if (intersectionArea === 0) return 0;
      
      var area1 = (box1.x2 - box1.x1) * (box1.y2 - box1.y1);
      var area2 = (box2.x2 - box2.x1) * (box2.y2 - box2.y1);
      var unionArea = area1 + area2 - intersectionArea;
      
      return intersectionArea / unionArea;
    }
    
    // Executar NMS
    function runNMS() {
      var tau = parseFloat(tauEl.value);
      var remaining = boxes.map(function(box, index) {
        return { box: box, originalIndex: index };
      });
      
      // Ordenar por score decrescente (estável)
      remaining.sort(function(a, b) {
        if (b.box.score !== a.box.score) {
          return b.box.score - a.box.score;
        }
        return a.originalIndex - b.originalIndex;
      });
      
      selectedBoxes = [];
      suppressedBoxes = [];
      var steps = [];
      
      while (remaining.length > 0) {
        var selected = remaining.shift();
        selectedBoxes.push(selected);
        
        steps.push({
          type: 'select',
          box: selected,
          remaining: remaining.slice()
        });
        
        var newRemaining = [];
        var suppressed = [];
        
        for (var i = 0; i < remaining.length; i++) {
          var iou = calculateIoU(selected.box, remaining[i].box);
          if (iou > tau) {
            suppressed.push({ box: remaining[i], iou: iou });
            suppressedBoxes.push(remaining[i]);
          } else {
            newRemaining.push(remaining[i]);
          }
        }
        
        if (suppressed.length > 0) {
          steps.push({
            type: 'suppress',
            box: selected,
            suppressed: suppressed,
            remaining: newRemaining.slice()
          });
        }
        
        remaining = newRemaining;
      }
      
      return steps;
    }
    
    // Renderizar visualização
    function render() {
      var tau = parseFloat(tauEl.value);
      nvEl.textContent = boxes.length;
      tauvEl.textContent = tau.toFixed(2);
      
      // Limpar canvas
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      
      // Desenhar todas as caixas
      boxes.forEach(function(box, index) {
        var isSelected = selectedBoxes.some(function(s) { return s.originalIndex === index; });
        var isSuppressed = suppressedBoxes.some(function(s) { return s.originalIndex === index; });
        
        var color = '#f5a623'; // candidata
        if (isSelected) color = '#4a90e2'; // selecionada
        if (isSuppressed) color = '#ff6b6b'; // suprimida
        
        drawBox(box, color, index);
      });
      
      // Atualizar resumo
      var summaryHTML = '';
      if (selectedBoxes.length > 0) {
        summaryHTML += '<b>Caixas selecionadas (em ordem):</b><br>';
        selectedBoxes.forEach(function(s) {
          summaryHTML += '#' + s.originalIndex + ' (score: ' + s.box.score.toFixed(4) + ')<br>';
        });
        summaryHTML += '<b>Total mantidas: ' + selectedBoxes.length + '</b>';
        summaryEl.innerHTML = summaryHTML;
      }
    }
    
    // Desenhar caixa
    function drawBox(box, color, index) {
      var scale = 2.0;
      var offsetX = 30;
      var offsetY = 30;
      
      var x = offsetX + box.x1 * scale;
      var y = offsetY + box.y1 * scale;
      var w = (box.x2 - box.x1) * scale;
      var h = (box.y2 - box.y1) * scale;
      
      // Desenhar caixa
      ctx.strokeStyle = color;
      ctx.lineWidth = 3;
      ctx.strokeRect(x, y, w, h);
      
      // Preenchimento translúcido
      ctx.fillStyle = color;
      ctx.globalAlpha = 0.2;
      ctx.fillRect(x, y, w, h);
      ctx.globalAlpha = 1;
      
      // Label
      ctx.fillStyle = color;
      ctx.font = 'bold 12px Arial';
      ctx.textAlign = 'center';
      ctx.fillText('#' + index, x + w/2, y - 5);
      
      // Score
      ctx.fillStyle = '#666';
      ctx.font = '10px Arial';
      ctx.fillText('score: ' + box.score.toFixed(2), x + w/2, y + h/2);
    }
    
    // Mostrar passo a passo
    function showSteps(steps) {
      var html = '';
      
      steps.forEach(function(step, index) {
        if (step.type === 'select') {
          html += '<div style="color:#4a90e2;font-weight:bold;margin-top:4px;">';
          html += 'Passo ' + (index + 1) + ': Selecionar caixa #' + step.box.originalIndex;
          html += ' (score: ' + step.box.box.score.toFixed(4) + ')';
          html += '</div>';
        } else if (step.type === 'suppress') {
          html += '<div style="color:#ff6b6b;margin-left:10px;">';
          html += '↳ Suprimir: ';
          step.suppressed.forEach(function(s, i) {
            if (i > 0) html += ', ';
            html += '#' + s.box.originalIndex;
            html += ' (IoU: ' + s.iou.toFixed(3) + ')';
          });
          html += '</div>';
        }
      });
      
      if (selectedBoxes.length > 0) {
        html += '<div style="color:#50e3c2;font-weight:bold;margin-top:8px;">';
        html += '✓ Resultado: ' + selectedBoxes.length + ' caixa(s) mantida(s)';
        html += '</div>';
      }
      
      stepsEl.innerHTML = html;
    }
    
    // Executar NMS
    function executeNMS() {
      var steps = runNMS();
      showSteps(steps);
      render();
      
      // Animação do botão
      runBtn.textContent = '✓ Executado!';
      runBtn.style.background = '#50e3c2';
      setTimeout(function() {
        runBtn.textContent = '▶️ Executar NMS';
        runBtn.style.background = '#2980b9';
      }, 1000);
    }
    
    // Event listeners
    runBtn.addEventListener('click', executeNMS);
    
    nEl.addEventListener('input', function() {
      var n = parseInt(nEl.value);
      var currentN = boxes.length;
      
      if (n > currentN) {
        for (var i = currentN; i < n; i++) {
          boxes.push({
            x1: 10 + i * 5,
            y1: 10 + i * 5,
            x2: 50 + i * 5,
            y2: 50 + i * 5,
            score: 0.9 - i * 0.1
          });
        }
      } else if (n < currentN) {
        boxes = boxes.slice(0, n);
      }
      
      selectedBoxes = [];
      suppressedBoxes = [];
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Caixas atualizadas: ' + boxes.length + '. Clique em "Executar NMS".';
    });
    
    tauEl.addEventListener('input', function() {
      tauvEl.textContent = parseFloat(tauEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Limiar atualizado. Clique em "Executar NMS".';
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('padrao');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0904');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.4:** Simulador EP09_04: IoU y Supresión de No-Máximos (NMS)


<figure id="fig-09-sim-ep0904">
  <img src="imagens/fig-09-sim-ep0904.png" alt=" Simulador EP09_04: IoU y Supresión de No-Máximos (NMS) " style="max-width:80%" />
  <figcaption><strong>Figura 9.4:</strong>  Simulador EP09_04: IoU y Supresión de No-Máximos (NMS) </figcaption>
</figure>

In [ ]:
%%writefile EP09_04.py
# Código Python


In [ ]:
TestSuite("EP09_04.py").run()

### EP09_05 🟠 Evaluación de Segmentación: IoU y Dice Pixel a Pixel

El Bloque 2 de la sección "Segmentación Semántica con Arquitectura U-Net" define, en pocas líneas, la función `iou_mascaras`, utilizada para medir la calidad de la línea base morfológica clásica (suavizado + Otsu + apertura) y, más adelante, de la propia U-Net entrenada. A diferencia del IoU del EP09_04 — calculado sobre **cajas delimitadoras** (regiones rectangulares descritas por cuatro números) —, el IoU de segmentación se calcula **pixel a pixel**: cada posición de la imagen se compara individualmente entre la máscara predicha y la máscara de referencia.

Se le ha encargado generalizar esta evaluación, implementando no solo el IoU pixel a pixel, sino también el **coeficiente de Dice**, otra métrica de superposición ampliamente utilizada en segmentación médica (incluso en la función `perda_dice`, mencionada en el mismo bloque del capítulo como base de la función de pérdida utilizada para entrenar la U-Net).

#### 📋 Directrices de Implementación

1. **Entrada:** Leer las dimensiones $H \times W$ de las máscaras.

2. **Máscara predicha:** Leer $H$ líneas con $W$ valores enteros (0 o 1) cada una — por ejemplo, la salida de una U-Net después de la umbralización en $0{,}5$ sobre la sigmoide, como en el Bloque 4 del capítulo.

3. **Máscara de referencia:** Leer otras $H$ líneas con $W$ valores enteros (0 o 1) cada una — el *ground truth*.

4. **Intersección y unión:** Considerando cada píxel como perteneciente al objeto cuando su valor es diferente de cero,
   $$
   \text{intersección} = \sum_{i,j} \mathbb{1}[P_{ij}=1 \wedge R_{ij}=1], \qquad
   \text{unión} = \sum_{i,j} \mathbb{1}[P_{ij}=1 \vee R_{ij}=1].
   $$

5. **IoU pixel a pixel:**
   $$
   \text{IoU} = \frac{\text{intersección}}{\text{unión}}.
   $$

6. **Coeficiente de Dice:**
   $$
   \text{Dice} = \frac{2 \cdot \text{intersección}}{|P| + |R|},
   $$
   donde $|P|$ y $|R|$ son el número total de píxeles de objeto en cada máscara.

7. **Convención para máscaras vacías:** si **ambas** máscaras no poseen ningún píxel de objeto (unión $= 0$ y $|P|+|R|=0$), considere la correspondencia trivialmente perfecta: $\text{IoU} = \text{Dice} = 1{,}0$.

8. **Salida:** Dos líneas, `IoU: X.XXXX` y `Dice: X.XXXX`, cada valor con 4 decimales.

#### 📌 Restricciones Computacionales

* **Cualquier valor no nulo cuenta como objeto:** trate valores diferentes de $0$ (no solo $1$) como pertenecientes a la máscara, replicando la comprobación `predita > 0` utilizada en `iou_mascaras` en el capítulo.
* **Mismas dimensiones:** las dos máscaras siempre poseen exactamente $H \times W$ elementos.
* **Convención de vacío:** aplique la regla del ítem 7 **solo** cuando ambas máscaras estén totalmente vacías; si solo una está vacía, la intersección es $0$ y el IoU/Dice resultante también será $0$.

#### 🧠 Fundamentación Teórica

| Elemento | Papel en la evaluación de segmentación |
|---|---|
| IoU pixel a pixel | Generaliza la métrica del EP09_04 para regiones de forma arbitraria — no solo rectángulos — comparando máscara predicha y referencia posición a posición |
| Coeficiente de Dice | Métrica relacionada con el IoU (siempre $\text{Dice} \ge \text{IoU}$), más sensible a pequeñas intersecciones y ampliamente utilizada como función de pérdida en segmentación (función `perda_dice` del capítulo) |
| Convención de máscaras vacías | Evita la división por cero y reconoce que "ningún objeto previsto, ningún objeto real" es, por definición, un acierto |
| Comparación clásico vs. U-Net | El capítulo usa exactamente este tipo de métrica para justificar, numéricamente, por qué la U-Net supera la línea base morfológica en escenarios de bajo contraste |

#### 🧩 Métodos del `morph.py` que pueden ayudar

* `mm.readImg(h, w, dtype='uint8')` — lee directamente cada máscara binaria $h \times w$ de la entrada estándar (los valores $0/1$ caben perfectamente en el tipo entero estándar).
* La propia función `iou_mascaras`, definida en el Bloque 2 de la sección de U-Net del capítulo (no forma parte del `morph.py`, sino del código del capítulo), es la inspiración directa de este ejercicio — vale la pena releer esas pocas líneas antes de programar.
* Para una extensión opcional (no exigida por este EP), `mm.connectedComponents` o `mm.label0` (vistos en el contexto de análisis de componentes conexos) permitirían etiquetar cada nódulo individualmente y calcular el IoU **por componente**, en lugar de sobre la máscara completa.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $H$ y $W$.
* Siguientes $H$ líneas: $W$ valores enteros (0 o 1) — máscara predicha.
* Siguientes $H$ líneas: $W$ valores enteros (0 o 1) — máscara de referencia.

**Salida:**

* Línea 1: `IoU: X.XXXX`.
* Línea 2: `Dice: X.XXXX`.

> ### 💡 Dica
>
> ##### 💡 Ejemplo Ilustrativo
>
> Considere una máscara predicha con un cuadrado $2\times2$ de píxeles activos y una referencia desplazada en una columna, superponiéndose en solo la mitad del área:
>
> ```
> Predicha         Referencia
> 0 0 0 0         0 0 0 0
> 0 1 1 0         0 0 1 1
> 0 1 1 0         0 0 1 1
> 0 0 0 0         0 0 0 0
> ```
>
> Intersección $=2$ píxeles, unión $=6$ píxeles ($4+4-2$), por lo tanto $\text{IoU}=2/6\approx0{,}3333$ y $\text{Dice}=2\cdot2/(4+4)=0{,}5000$ — observe que el Dice es siempre igual o mayor que el IoU para la misma superposición.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 4 4<br>0 0 0 0<br>0 1 1 0<br>0 1 1 0<br>0 0 0 0<br>0 0 0 0<br>0 0 1 1<br>0 0 1 1<br>0 0 0 0 | IoU: 0.3333<br>Dice: 0.5000 | Máscaras $4\times4$ con superposición parcial de 2 píxeles. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0905" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: IoU y Dice Píxel a Píxel</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟠 Segmentación</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Intersección</b> (TP)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Solo predicha</b> (FP)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Solo referencia</b> (FN)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f0f0f0;border:2px solid #ccc;border-radius:2px;display:inline-block;"></span>
        <b>Fondo</b> (TN)
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:flex;gap:10px;margin-bottom:8px;flex-wrap:wrap;">
        <div style="flex:1;min-width:120px;">
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Dimensiones</label>
            <span id="ep0905_dim_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">5×5</span>
          </div>
          <input id="ep0905_dim" style="width:100%;accent-color:#2980b9;height:4px;" max="8" min="3" step="1" type="range" value="5">
        </div>
        <div style="flex:1;min-width:120px;">
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Ejemplo</label>
            <span id="ep0905_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Cuadrado</span>
          </div>
          <select id="ep0905_example" style="width:100%;padding:4px;border:1px solid #ccc;border-radius:3px;font-size:10px;">
            <option value="quadrado">Cuadrado 2×2</option>
            <option value="deslocado">Desplazado</option>
            <option value="perfeito">Perfecto</option>
            <option value="vazio">Máscaras Vacías</option>
            <option value="parcial">Superposición Parcial</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;gap:8px;">
          <button id="ep0905_clear_btn" style="background:#666;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;">🗑️ Limpiar</button>
          <button id="ep0905_random_btn" style="background:#f5a623;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;">🎲 Aleatorio</button>
        </div>
      </div>
      
      <!-- Grids de máscaras -->
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:10px;">
        <div>
          <div style="font-weight:bold;font-size:11px;color:#333;margin-bottom:4px;text-align:center;">
            🔵 Máscara Predicha
          </div>
          <div id="ep0905_pred_grid" style="display:flex;justify-content:center;overflow-x:auto;"></div>
        </div>
        <div>
          <div style="font-weight:bold;font-size:11px;color:#333;margin-bottom:4px;text-align:center;">
            🟡 Máscara de Referencia
          </div>
          <div id="ep0905_ref_grid" style="display:flex;justify-content:center;overflow-x:auto;"></div>
        </div>
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas da visualização -->
      <div style="background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;">
        <div style="color:white;font-weight:bold;font-size:13px;text-align:center;margin-bottom:8px;">
          🎯 Comparación Visual
        </div>
        <canvas id="ep0905_canvas" style="width:100%;height:220px;display:block;background:white;border-radius:5px;"></canvas>
      </div>
      
      <!-- Métricas -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:280px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📊 Cálculos y Fórmulas
        </div>
        <div id="ep0905_metrics" style="font-size:10px;line-height:1.8;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resumo final -->
    <div id="ep0905_summary" style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);border-radius:8px;padding:12px;font-family:monospace;font-size:14px;color:white;line-height:1.8;text-align:center;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var dimEl = root.querySelector('#ep0905_dim');
    var dimvEl = root.querySelector('#ep0905_dim_v');
    var exampleEl = root.querySelector('#ep0905_example');
    var predGridEl = root.querySelector('#ep0905_pred_grid');
    var refGridEl = root.querySelector('#ep0905_ref_grid');
    var canvas = root.querySelector('#ep0905_canvas');
    var ctx = canvas.getContext('2d');
    var metricsEl = root.querySelector('#ep0905_metrics');
    var summaryEl = root.querySelector('#ep0905_summary');
    var clearBtn = root.querySelector('#ep0905_clear_btn');
    var randomBtn = root.querySelector('#ep0905_random_btn');
    
    // Estado
    var predMask = [];
    var refMask = [];
    var H = 5;
    var W = 5;
    
    // Exemplos pré-definidos
    var examples = {
      quadrado: {
        pred: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      deslocado: {
        pred: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      perfeito: {
        pred: [[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]],
        ref: [[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]]
      },
      vazio: {
        pred: [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      parcial: {
        pred: [[0,0,0,0,0],[0,1,1,1,0],[0,1,1,1,0],[0,1,1,1,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,0,0,0]]
      }
    };
    
    // Ajustar tamanho do canvas
    function resizeCanvas() {
      var rect = canvas.getBoundingClientRect();
      canvas.width = rect.width;
      canvas.height = rect.height;
    }
    
    // Carregar exemplo
    function loadExample(name) {
      var example = examples[name];
      predMask = example.pred.map(function(row) { return row.slice(); });
      refMask = example.ref.map(function(row) { return row.slice(); });
      H = predMask.length;
      W = predMask[0].length;
      dimEl.value = H;
      dimvEl.textContent = H + '×' + W;
      generateGrids();
      render();
    }
    
    // Gerar grids clicáveis
    function generateGrids() {
      var predHTML = '<table style="border-collapse:collapse;">';
      var refHTML = '<table style="border-collapse:collapse;">';
      
      for (var i = 0; i < H; i++) {
        predHTML += '<tr>';
        refHTML += '<tr>';
        for (var j = 0; j < W; j++) {
          predHTML += '<td data-type="pred" data-i="' + i + '" data-j="' + j + '" style="width:28px;height:28px;background:' + (predMask[i][j] ? '#4a90e2' : 'white') + ';border:1px solid #999;cursor:pointer;text-align:center;font-size:10px;color:' + (predMask[i][j] ? 'white' : '#999') + ';">' + (predMask[i][j] ? '1' : '0') + '</td>';
          refHTML += '<td data-type="ref" data-i="' + i + '" data-j="' + j + '" style="width:28px;height:28px;background:' + (refMask[i][j] ? '#f5a623' : 'white') + ';border:1px solid #999;cursor:pointer;text-align:center;font-size:10px;color:' + (refMask[i][j] ? 'white' : '#999') + ';">' + (refMask[i][j] ? '1' : '0') + '</td>';
        }
        predHTML += '</tr>';
        refHTML += '</tr>';
      }
      
      predHTML += '</table>';
      refHTML += '</table>';
      
      predGridEl.innerHTML = predHTML;
      refGridEl.innerHTML = refHTML;
      
      // Adicionar event listeners
      predGridEl.querySelectorAll('td[data-type="pred"]').forEach(function(cell) {
        cell.addEventListener('click', function() {
          var i = parseInt(cell.getAttribute('data-i'));
          var j = parseInt(cell.getAttribute('data-j'));
          predMask[i][j] = predMask[i][j] ? 0 : 1;
          generateGrids();
          render();
        });
      });
      
      refGridEl.querySelectorAll('td[data-type="ref"]').forEach(function(cell) {
        cell.addEventListener('click', function() {
          var i = parseInt(cell.getAttribute('data-i'));
          var j = parseInt(cell.getAttribute('data-j'));
          refMask[i][j] = refMask[i][j] ? 0 : 1;
          generateGrids();
          render();
        });
      });
    }
    
    // Calcular métricas
    function calculateMetrics() {
      var TP = 0, FP = 0, FN = 0, TN = 0;
      
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          var p = predMask[i][j] ? 1 : 0;
          var r = refMask[i][j] ? 1 : 0;
          
          if (p && r) TP++;
          else if (p && !r) FP++;
          else if (!p && r) FN++;
          else TN++;
        }
      }
      
      var intersection = TP;
      var union = TP + FP + FN;
      var predCount = TP + FP;
      var refCount = TP + FN;
      
      var iou, dice;
      
      if (union === 0) {
        iou = 1.0;
        dice = 1.0;
      } else {
        iou = intersection / union;
        dice = (predCount + refCount === 0) ? 1.0 : (2 * intersection) / (predCount + refCount);
      }
      
      return { TP, FP, FN, TN, intersection, union, predCount, refCount, iou, dice };
    }
    
    // Renderizar visualização
    function render() {
      resizeCanvas();
      var m = calculateMetrics();
      
      // Limpar canvas
      ctx.fillStyle = 'white';
      ctx.fillRect(0, 0, canvas.width, canvas.height);
      
      var cellSize = Math.min(35, (canvas.width - 40) / W);
      var offsetX = (canvas.width - W * cellSize) / 2;
      var offsetY = (canvas.height - H * cellSize) / 2;
      
      // Desenhar grid
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          var p = predMask[i][j] ? 1 : 0;
          var r = refMask[i][j] ? 1 : 0;
          
          var color = '#f0f0f0';
          if (p && r) color = '#4a90e2';
          else if (p && !r) color = '#ff6b6b';
          else if (!p && r) color = '#f5a623';
          
          var x = offsetX + j * cellSize;
          var y = offsetY + i * cellSize;
          
          ctx.fillStyle = color;
          ctx.fillRect(x, y, cellSize - 2, cellSize - 2);
          ctx.strokeStyle = '#999';
          ctx.lineWidth = 1;
          ctx.strokeRect(x, y, cellSize - 2, cellSize - 2);
        }
      }
      
      // Fórmulas e cálculos
      var html = '';
      html += '<div style="margin-bottom:6px;"><b>1. Contagem de pixels:</b></div>';
      html += '<div style="color:#4a90e2;">TP (interseção) = ' + m.TP + '</div>';
      html += '<div style="color:#ff6b6b;">FP (só predita) = ' + m.FP + '</div>';
      html += '<div style="color:#f5a623;">FN (só referência) = ' + m.FN + '</div>';
      html += '<div style="color:#999;">TN (fundo) = ' + m.TN + '</div>';
      
      html += '<div style="margin-top:8px;margin-bottom:4px;"><b>2. Interseção e União:</b></div>';
      html += '<div>Interseção = TP = <b>' + m.intersection + '</b></div>';
      html += '<div>União = TP + FP + FN = ' + m.TP + ' + ' + m.FP + ' + ' + m.FN + ' = <b>' + m.union + '</b></div>';
      html += '<div>|P| = TP + FP = ' + m.TP + ' + ' + m.FP + ' = <b>' + m.predCount + '</b></div>';
      html += '<div>|R| = TP + FN = ' + m.TP + ' + ' + m.FN + ' = <b>' + m.refCount + '</b></div>';
      
      html += '<div style="margin-top:8px;margin-bottom:4px;"><b>3. Fórmulas:</b></div>';
      
      if (m.union === 0) {
        html += '<div style="color:#888;">IoU = 1.0 (máscaras vazias)</div>';
        html += '<div style="color:#888;">Dice = 1.0 (máscaras vazias)</div>';
      } else {
        html += '<div>IoU = Interseção / União = ' + m.intersection + ' / ' + m.union + ' = <b style="color:#2980b9;">' + m.iou.toFixed(4) + '</b></div>';
        html += '<div>Dice = 2·Interseção / (|P| + |R|) = 2·' + m.intersection + ' / (' + m.predCount + ' + ' + m.refCount + ') = ' + (2 * m.intersection) + ' / ' + (m.predCount + m.refCount) + ' = <b style="color:#50e3c2;">' + m.dice.toFixed(4) + '</b></div>';
      }
      
      metricsEl.innerHTML = html;
      
      // Resumo final
      summaryEl.innerHTML = '<b>IoU: ' + m.iou.toFixed(4) + '</b> &nbsp;&nbsp;|&nbsp;&nbsp; <b>Dice: ' + m.dice.toFixed(4) + '</b>';
    }
    
    // Limpar máscaras
    function clearMasks() {
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          predMask[i][j] = 0;
          refMask[i][j] = 0;
        }
      }
      generateGrids();
      render();
    }
    
    // Gerar máscaras aleatórias
    function randomMasks() {
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          predMask[i][j] = Math.random() > 0.5 ? 1 : 0;
          refMask[i][j] = Math.random() > 0.5 ? 1 : 0;
        }
      }
      generateGrids();
      render();
    }
    
    // Event listeners
    clearBtn.addEventListener('click', clearMasks);
    randomBtn.addEventListener('click', randomMasks);
    
    dimEl.addEventListener('input', function() {
      H = parseInt(dimEl.value);
      W = H;
      dimvEl.textContent = H + '×' + W;
      
      var newPred = [];
      var newRef = [];
      for (var i = 0; i < H; i++) {
        newPred.push([]);
        newRef.push([]);
        for (var j = 0; j < W; j++) {
          newPred[i].push(i < predMask.length && j < predMask[0].length ? predMask[i][j] : 0);
          newRef[i].push(i < refMask.length && j < refMask[0].length ? refMask[i][j] : 0);
        }
      }
      predMask = newPred;
      refMask = newRef;
      generateGrids();
      render();
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('quadrado');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0905');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.5:** Simulador EP09_05: Evaluación de Segmentación — IoU y Dice Píxel a Píxel


<figure id="fig-09-sim-ep0905">
  <img src="imagens/fig-09-sim-ep0905.png" alt=" Simulador EP09_05: Evaluación de Segmentación — IoU y Dice Píxel a Píxel " style="max-width:80%" />
  <figcaption><strong>Figura 9.5:</strong>  Simulador EP09_05: Evaluación de Segmentación — IoU y Dice Píxel a Píxel </figcaption>
</figure>

In [ ]:
%%writefile EP09_05.py
# Código Python en español


In [ ]:
TestSuite("EP09_05.py").run()

### EP09_06 🔴 *Pipeline* Integrado: De la Detección a la Medición del Mundo Real

Este ejercicio final integra los dos ejercicios de detección y el principio de **fotogrametría** presentado en la sección "Fotogrametría y Referencia de Escala" — exactamente el mismo cálculo implementado en la figura de medición por referencia de escala de este capítulo. El escenario reproduce una situación realista: un detector (Faster R-CNN o YOLO) genera **varias cajas candidatas superpuestas** para el mismo objeto de interés; tras filtrarlas por NMS, la caja superviviente de mayor confianza se usa, junto con una caja de referencia de ancho real conocido (como la tarjeta de $8{,}56$ cm), para estimar las dimensiones reales del objeto detectado.

#### 📋 Directrices de Implementación

1. **Referencia conocida:** Leer el valor real $L_{ref}$ (ancho real del objeto de referencia, en cm) y, a continuación, los cuatro reales $x_1\ y_1\ x_2\ y_2$ de su caja delimitadora en píxeles (ya conocida, sin necesidad de detección).
2. **Candidatas del objeto a medir:** Leer el entero $N$ (número de cajas candidatas producidas por el detector para el objeto de interés) y el umbral real $\tau$; a continuación, leer las $N$ líneas de cajas candidatas, cada una con $x_1\ y_1\ x_2\ y_2\ \text{score}$.
3. **Etapa 1 — NMS:** Aplicar exactamente el algoritmo de Supresión de No Máximos del EP09_04 a las $N$ cajas candidatas, usando el umbral $\tau$, para eliminar detecciones redundantes del mismo objeto.
4. **Etapa 2 — Selección de la caja final:** Tras el NMS, la caja de mayor `score` entre las mantenidas es la detección final del objeto (la entrada garantiza que todas las cajas candidatas corresponden a un único objeto físico, por lo que la primera caja seleccionada por el NMS ya es el resultado final).
5. **Etapa 3 — Medición por referencia de escala:** Calcular la razón $\text{cm/pixel} = L_{ref} / \text{ancho de la referencia en píxeles}$ y aplicarla tanto al ancho como a la altura (en píxeles) de la caja final del objeto, obteniendo sus dimensiones reales estimadas en centímetros.
6. **Salida:** Primero, una línea por caja mantenida tras el NMS (mismo formato del EP09_04): `índice score`. A continuación, la línea `Total mantenidas: X`. Finalmente, la línea `Objeto: L x A cm`, donde $L$ y $A$ son el ancho y la altura estimados del objeto, cada uno con 2 decimales.

#### 📌 Restricciones Computacionales

* **Reutilizar el NMS del EP09_04** íntegramente — misma regla de desempate, mismo criterio de supresión ($\text{IoU} > \tau$).
* **La referencia no pasa por NMS:** su caja se da directamente, sin candidatas competidoras.
* **Razón única para ancho y altura:** así como en la figura de fotogrametría del capítulo, la misma razón cm/pixel (derivada del ancho de la referencia) se aplica tanto al ancho como a la altura del objeto — no hay calibración vertical separada.

#### 🧠 Fundamentación Teórica

| Etapa | Concepto del capítulo |
|---|---|
| Múltiples cajas candidatas | Salida bruta de un detector como el Faster R-CNN o el YOLO, antes del posprocesamiento |
| NMS (EP09_04) | Filtra las detecciones redundantes, preservando solo la más confiable para el objeto |
| Referencia de escala conocida | Mismo principio de la tarjeta de $8{,}56$ cm usado en la sección "Fotogrametría y Referencia de Escala" |
| Conversión píxel → centímetro | Regla de tres simple: $\text{cm/pixel} = L_{ref} / w_{ref\_px}$, aplicada a la caja final del objeto |

#### 🧩 Métodos del `morph.py` que pueden ayudar

* `mm.IoU(boxA, boxB)` — la misma función sugerida en el EP09_04, aquí reutilizada dentro de la etapa de NMS de este *pipeline* integrado (recuerde la conversión de formato: $w = x_2-x_1$, $h = y_2-y_1$).
* Si ya resolvió el EP09_04 encapsulando el NMS en una función propia, este es el momento ideal de **reutilizar ese código** — la integración de módulos ya probados individualmente es exactamente la práctica de ingeniería que este ejercicio quiere reforzar.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Real $L_{ref}$.
* Línea 2: $x_1\ y_1\ x_2\ y_2$ de la caja de referencia.
* Línea 3: Entero $N$ y real $\tau$.
* Siguientes $N$ líneas: $x_1\ y_1\ x_2\ y_2\ \text{score}$ de las cajas candidatas del objeto.

**Salida:**

* Una línea por caja mantenida tras el NMS: `índice score`.
* Línea siguiente: `Total mantenidas: X`.
* Última línea: `Objeto: L x A cm`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 8.56<br>30 200 170 288<br>3 0.5<br>250 100 470 250 0.92<br>255 105 468 245 0.88<br>600 600 650 650 0.40 | 0 0.9200<br>2 0.4000<br>Total mantenidas: 2<br>Objeto: 13.45 x 9.17 cm | La caja 1 se suprime por superponerse fuertemente a la caja 0; la detección final del objeto es la caja 0. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0906" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Pipeline Integrado — Detección a Medición</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 Fotogrametría</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Caja seleccionada</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Caja suprimida</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:2px;display:inline-block;"></span>
        <b>Referencia</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Objeto final</b>
      </span>
    </div>
    
    <!-- Controles -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">L_ref (cm)</label>
            <span id="ep0906_lref_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">8.56</span>
          </div>
          <input id="ep0906_lref" style="width:100%;accent-color:#2980b9;height:4px;" max="20" min="1" step="0.01" type="range" value="8.56">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Umbral τ</label>
            <span id="ep0906_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">0.50</span>
          </div>
          <input id="ep0906_tau" style="width:100%;accent-color:#2980b9;height:4px;" max="1.0" min="0.1" step="0.05" type="range" value="0.5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Ejemplo</label>
            <span id="ep0906_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Estándar</span>
          </div>
          <select id="ep0906_example" style="width:100%;padding:4px;border:1px solid #ccc;border-radius:3px;font-size:10px;">
            <option value="padrao">Ejemplo Estándar</option>
            <option value="multiplos">Múltiples Objetos</option>
            <option value="agrupado">Cajas Agrupadas</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;justify-content:center;">
          <button id="ep0906_run_btn" style="background:#2980b9;color:white;border:none;padding:8px 16px;border-radius:16px;cursor:pointer;font-size:11px;font-weight:bold;transition:all 0.3s;">
            ▶️ Procesar Pipeline
          </button>
        </div>
      </div>
      
      <!-- Caixas configuráveis -->
      <div id="ep0906_boxes_config" style="display:flex;flex-wrap:wrap;gap:6px;font-size:9px;">
        <!-- Gerado dinamicamente -->
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:3fr 2fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas -->
      <div style="background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;min-height:350px;">
        <div style="color:white;font-weight:bold;font-size:13px;text-align:center;margin-bottom:8px;">
          🎯 Visualización del Pipeline
        </div>
        <canvas id="ep0906_canvas" style="width:100%;height:300px;display:block;background:white;border-radius:5px;"></canvas>
      </div>
      
      <!-- Passo a passo -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:350px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📋 Pipeline Paso a Paso
        </div>
        <div id="ep0906_steps" style="font-size:10px;line-height:1.8;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resultado final -->
    <div id="ep0906_summary" style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);border-radius:8px;padding:12px;font-family:monospace;font-size:14px;color:white;line-height:1.8;text-align:center;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var lrefEl = root.querySelector('#ep0906_lref');
    var lrefvEl = root.querySelector('#ep0906_lref_v');
    var tauEl = root.querySelector('#ep0906_tau');
    var tauvEl = root.querySelector('#ep0906_tau_v');
    var exampleEl = root.querySelector('#ep0906_example');
    var boxesConfigEl = root.querySelector('#ep0906_boxes_config');
    var canvas = root.querySelector('#ep0906_canvas');
    var ctx = canvas.getContext('2d');
    var stepsEl = root.querySelector('#ep0906_steps');
    var summaryEl = root.querySelector('#ep0906_summary');
    var runBtn = root.querySelector('#ep0906_run_btn');
    
    // Estado
    var refBox = { x1: 30, y1: 200, x2: 170, y2: 288 };
    var boxes = [];
    var selectedBoxes = [];
    var suppressedBoxes = [];
    var finalBox = null;
    var cmPerPixel = 0;
    
    // Exemplos
    var examples = {
      padrao: {
        refBox: { x1: 30, y1: 200, x2: 170, y2: 288 },
        boxes: [
          { x1: 250, y1: 100, x2: 470, y2: 250, score: 0.92 },
          { x1: 255, y1: 105, x2: 468, y2: 245, score: 0.88 },
          { x1: 600, y1: 600, x2: 650, y2: 650, score: 0.40 }
        ]
      },
      multiplos: {
        refBox: { x1: 20, y1: 50, x2: 100, y2: 130 },
        boxes: [
          { x1: 200, y1: 150, x2: 350, y2: 280, score: 0.85 },
          { x1: 210, y1: 160, x2: 360, y2: 290, score: 0.75 },
          { x1: 400, y1: 300, x2: 550, y2: 420, score: 0.70 },
          { x1: 410, y1: 310, x2: 560, y2: 430, score: 0.65 }
        ]
      },
      agrupado: {
        refBox: { x1: 50, y1: 50, x2: 150, y2: 150 },
        boxes: [
          { x1: 300, y1: 200, x2: 500, y2: 350, score: 0.95 },
          { x1: 310, y1: 210, x2: 490, y2: 340, score: 0.90 },
          { x1: 320, y1: 220, x2: 480, y2: 330, score: 0.85 },
          { x1: 330, y1: 230, x2: 470, y2: 320, score: 0.80 }
        ]
      }
    };
    
    // Ajustar canvas
    function resizeCanvas() {
      var rect = canvas.getBoundingClientRect();
      canvas.width = rect.width;
      canvas.height = rect.height;
    }
    
    // Carregar exemplo
    function loadExample(name) {
      var example = examples[name];
      refBox = JSON.parse(JSON.stringify(example.refBox));
      boxes = JSON.parse(JSON.stringify(example.boxes));
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Processar Pipeline" para executar.';
      summaryEl.innerHTML = 'Aguardando processamento...';
    }
    
    // Gerar configuração das caixas
    function generateBoxesConfig() {
      var html = '';
      html += '<div style="background:#e8f5e9;border:1px solid #a5d6a7;border-radius:6px;padding:4px 6px;">';
      html += '<b>Referência:</b> ';
      html += '<input type="number" id="ep0906_ref_x1" value="' + refBox.x1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_y1" value="' + refBox.y1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_x2" value="' + refBox.x2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_y2" value="' + refBox.y2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '</div>';
      
      boxes.forEach(function(box, i) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:4px 6px;">';
        html += '<b>#' + i + ':</b> ';
        html += '<input type="number" id="ep0906_x1_' + i + '" value="' + box.x1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_y1_' + i + '" value="' + box.y1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_x2_' + i + '" value="' + box.x2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_y2_' + i + '" value="' + box.y2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_score_' + i + '" value="' + box.score + '" step="0.05" min="0" max="1" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '</div>';
      });
      
      boxesConfigEl.innerHTML = html;
      
      // Event listeners para referência
      ['x1', 'y1', 'x2', 'y2'].forEach(function(field) {
        var input = root.querySelector('#ep0906_ref_' + field);
        if (input) {
          input.addEventListener('input', function() {
            refBox[field] = parseFloat(input.value) || 0;
            render();
          });
        }
      });
      
      // Event listeners para caixas
      boxes.forEach(function(box, i) {
        ['x1', 'y1', 'x2', 'y2', 'score'].forEach(function(field) {
          var input = root.querySelector('#ep0906_' + field + '_' + i);
          if (input) {
            input.addEventListener('input', function() {
              boxes[i][field] = parseFloat(input.value) || 0;
              selectedBoxes = [];
              suppressedBoxes = [];
              finalBox = null;
              render();
            });
          }
        });
      });
    }
    
    // Calcular IoU
    function calculateIoU(box1, box2) {
      var x1 = Math.max(box1.x1, box2.x1);
      var y1 = Math.max(box1.y1, box2.y1);
      var x2 = Math.min(box1.x2, box2.x2);
      var y2 = Math.min(box1.y2, box2.y2);
      
      var intersectionArea = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
      if (intersectionArea === 0) return 0;
      
      var area1 = (box1.x2 - box1.x1) * (box1.y2 - box1.y1);
      var area2 = (box2.x2 - box2.x1) * (box2.y2 - box2.y1);
      var unionArea = area1 + area2 - intersectionArea;
      
      return intersectionArea / unionArea;
    }
    
    // Executar pipeline
    function runPipeline() {
      var tau = parseFloat(tauEl.value);
      var lref = parseFloat(lrefEl.value);
      
      // Etapa 1: NMS
      var remaining = boxes.map(function(box, index) {
        return { box: box, originalIndex: index };
      });
      
      remaining.sort(function(a, b) {
        if (b.box.score !== a.box.score) return b.box.score - a.box.score;
        return a.originalIndex - b.originalIndex;
      });
      
      selectedBoxes = [];
      suppressedBoxes = [];
      var steps = [];
      
      steps.push('<div style="font-weight:bold;color:#333;">Etapa 1: NMS</div>');
      
      while (remaining.length > 0) {
        var selected = remaining.shift();
        selectedBoxes.push(selected);
        
        steps.push('<div style="color:#4a90e2;">Selecionar caixa #' + selected.originalIndex + ' (score: ' + selected.box.score.toFixed(4) + ')</div>');
        
        var newRemaining = [];
        for (var i = 0; i < remaining.length; i++) {
          var iou = calculateIoU(selected.box, remaining[i].box);
          if (iou > tau) {
            suppressedBoxes.push(remaining[i]);
            steps.push('<div style="color:#ff6b6b;margin-left:10px;">↳ Suprimir #' + remaining[i].originalIndex + ' (IoU: ' + iou.toFixed(3) + ')</div>');
          } else {
            newRemaining.push(remaining[i]);
          }
        }
        remaining = newRemaining;
      }
      
      steps.push('<div style="margin-top:4px;">Total mantidas: <b>' + selectedBoxes.length + '</b></div>');
      
      // Etapa 2: Seleção da caixa final
      if (selectedBoxes.length > 0) {
        finalBox = selectedBoxes[0];
        steps.push('<div style="margin-top:8px;font-weight:bold;color:#333;">Etapa 2: Caixa Final</div>');
        steps.push('<div>Caixa selecionada: #' + finalBox.originalIndex + ' (score: ' + finalBox.box.score.toFixed(4) + ')</div>');
      }
      
      // Etapa 3: Medição
      steps.push('<div style="margin-top:8px;font-weight:bold;color:#333;">Etapa 3: Medição por Referência</div>');
      
      var refWidthPx = refBox.x2 - refBox.x1;
      cmPerPixel = lref / refWidthPx;
      
      steps.push('<div>Largura da referência: ' + refWidthPx + ' pixels</div>');
      steps.push('<div>cm/pixel = ' + lref + ' / ' + refWidthPx + ' = <b>' + cmPerPixel.toFixed(6) + '</b></div>');
      
      if (finalBox) {
        var objWidthPx = finalBox.box.x2 - finalBox.box.x1;
        var objHeightPx = finalBox.box.y2 - finalBox.box.y1;
        var objWidthCm = objWidthPx * cmPerPixel;
        var objHeightCm = objHeightPx * cmPerPixel;
        
        steps.push('<div>Largura do objeto: ' + objWidthPx + ' px × ' + cmPerPixel.toFixed(6) + ' = <b>' + objWidthCm.toFixed(2) + ' cm</b></div>');
        steps.push('<div>Altura do objeto: ' + objHeightPx + ' px × ' + cmPerPixel.toFixed(6) + ' = <b>' + objHeightCm.toFixed(2) + ' cm</b></div>');
        
        stepsEl.innerHTML = steps.join('');
        
        summaryEl.innerHTML = '<b>Objeto: ' + objWidthCm.toFixed(2) + ' x ' + objHeightCm.toFixed(2) + ' cm</b>';
      } else {
        stepsEl.innerHTML = steps.join('');
        summaryEl.innerHTML = 'Nenhum objeto detectado.';
      }
      
      render();
    }
    
    // Renderizar visualização
    function render() {
      resizeCanvas();
      
      // Limpar canvas
      ctx.fillStyle = 'white';
      ctx.fillRect(0, 0, canvas.width, canvas.height);
      
      var scale = Math.min(canvas.width / 700, canvas.height / 700);
      var offsetX = 20;
      var offsetY = 20;
      
      // Desenhar caixa de referência
      drawBoxOnCanvas(refBox, '#50e3c2', 'Ref', scale, offsetX, offsetY);
      
      // Desenhar caixas
      boxes.forEach(function(box, index) {
        var isSelected = selectedBoxes.some(function(s) { return s.originalIndex === index; });
        var isSuppressed = suppressedBoxes.some(function(s) { return s.originalIndex === index; });
        var isFinal = finalBox && finalBox.originalIndex === index;
        
        var color = '#f5a623';
        var label = '#' + index;
        
        if (isFinal) {
          color = '#f5a623';
          label = '#' + index + ' ✓';
        } else if (isSelected) {
          color = '#4a90e2';
        } else if (isSuppressed) {
          color = '#ff6b6b';
          label = '#' + index + ' ✗';
        }
        
        drawBoxOnCanvas(box, color, label, scale, offsetX, offsetY);
      });
      
      // Desenhar linhas de medição
      if (finalBox && cmPerPixel > 0) {
        var refWidthPx = refBox.x2 - refBox.x1;
        var objWidthPx = finalBox.box.x2 - finalBox.box.x1;
        var objHeightPx = finalBox.box.y2 - finalBox.box.y1;
        var objWidthCm = objWidthPx * cmPerPixel;
        var objHeightCm = objHeightPx * cmPerPixel;
        
        // Linha de largura do objeto
        var y = offsetY + finalBox.box.y1 * scale - 10;
        var x1 = offsetX + finalBox.box.x1 * scale;
        var x2 = offsetX + finalBox.box.x2 * scale;
        
        ctx.strokeStyle = '#f5a623';
        ctx.lineWidth = 2;
        ctx.beginPath();
        ctx.moveTo(x1, y);
        ctx.lineTo(x2, y);
        ctx.stroke();
        
        ctx.fillStyle = '#f5a623';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(objWidthCm.toFixed(2) + ' cm', (x1 + x2) / 2, y - 3);
      }
    }
    
    // Desenhar caixa no canvas
    function drawBoxOnCanvas(box, color, label, scale, offsetX, offsetY) {
      var x = offsetX + box.x1 * scale;
      var y = offsetY + box.y1 * scale;
      var w = (box.x2 - box.x1) * scale;
      var h = (box.y2 - box.y1) * scale;
      
      ctx.strokeStyle = color;
      ctx.lineWidth = 3;
      ctx.strokeRect(x, y, w, h);
      
      ctx.fillStyle = color;
      ctx.globalAlpha = 0.2;
      ctx.fillRect(x, y, w, h);
      ctx.globalAlpha = 1;
      
      ctx.fillStyle = color;
      ctx.font = 'bold 11px Arial';
      ctx.textAlign = 'center';
      ctx.fillText(label, x + w/2, y - 5);
    }
    
    // Event listeners
    runBtn.addEventListener('click', function() {
      runPipeline();
      runBtn.textContent = '✓ Processado!';
      runBtn.style.background = '#50e3c2';
      setTimeout(function() {
        runBtn.textContent = '▶️ Processar Pipeline';
        runBtn.style.background = '#2980b9';
      }, 1000);
    });
    
    lrefEl.addEventListener('input', function() {
      lrefvEl.textContent = parseFloat(lrefEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      render();
    });
    
    tauEl.addEventListener('input', function() {
      tauvEl.textContent = parseFloat(tauEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      render();
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('padrao');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0906');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.6:** Simulador EP09_06: Pipeline Integrado — Detección a la Medición del Mundo Real


<figure id="fig-09-sim-ep0906">
  <img src="imagens/fig-09-sim-ep0906.png" alt=" Simulador EP09_06: Pipeline Integrado — Detección a la Medición del Mundo Real " style="max-width:80%" />
  <figcaption><strong>Figura 9.6:</strong>  Simulador EP09_06: Pipeline Integrado — Detección a la Medición del Mundo Real </figcaption>
</figure>

In [ ]:
%%writefile EP09_06.py
# Código Python

In [ ]:
TestSuite("EP09_06.py").run()